# código piloto

In [25]:
import os
import csv
import json
import time
from pathlib import Path
from datetime import datetime

import requests

# =========================
# CONFIG
# =========================
INPUT_FILE = "df_1000_api.txt"   # ajustá la ruta si hiciera falta
N_ARTISTS = 20
SLEEP_SECONDS = 1.1

# NO hardcodear el token en el notebook
#REFRESH_TOKEN = os.getenv("CHARTMETRIC_REFRESH_TOKEN")
REFRESH_TOKEN = "112E1dROGObsUx9hWKnrOxLQWrjDZ9YXwoByI9ynpN7QjtFY2Lq1E3uW7zvuIrpY"
if not REFRESH_TOKEN:
    raise ValueError("No encontré CHARTMETRIC_REFRESH_TOKEN en variables de entorno.")

BASE_DIR = Path("chartmetric_metadata_pilot")
RAW_DIR = BASE_DIR / "raw_json"
BASE_DIR.mkdir(exist_ok=True)
RAW_DIR.mkdir(exist_ok=True)

SUMMARY_CSV = BASE_DIR / "download_summary.csv"
FLAT_CSV = BASE_DIR / "metadata_flat.csv"

# =========================
# HELPERS
# =========================
def safe_name(name: str) -> str:
    return (
        str(name).strip().lower()
        .replace(" ", "_")
        .replace("/", "_")
        .replace("\\", "_")
        .replace(":", "_")
        .replace("?", "_")
        .replace("*", "_")
        .replace('"', "_")
        .replace("<", "_")
        .replace(">", "_")
        .replace("|", "_")
    )

def get_access_token(refresh_token: str) -> str:
    resp = requests.post(
        "https://api.chartmetric.com/api/token",
        json={"refreshtoken": refresh_token},
        timeout=60
    )
    resp.raise_for_status()
    data = resp.json()
    if "token" not in data:
        raise ValueError(f"No vino 'token' en la respuesta: {data}")
    return data["token"]

def build_headers(access_token: str) -> dict:
    return {
        "Authorization": f"Bearer {access_token}",
        "X-Accept-Partial-Data": "true"
    }

def read_artist_file(path: str):
    """
    Intenta detectar delimitador automáticamente.
    Espera columnas:
      - chartmetric_id
      - artist_name
      - chartmetric_url
    """
    with open(path, "r", encoding="utf-8-sig", newline="") as f:
        sample = f.read(5000)
        f.seek(0)

        try:
            dialect = csv.Sniffer().sniff(sample, delimiters=",\t;|")
            delimiter = dialect.delimiter
        except csv.Error:
            delimiter = "\t"  # fallback conservador

        reader = csv.DictReader(f, delimiter=delimiter)
        rows = list(reader)

    required = {"chartmetric_id", "artist_name", "chartmetric_url"}
    if not rows:
        raise ValueError("El archivo no tiene filas legibles.")
    if not required.issubset(set(rows[0].keys())):
        raise ValueError(
            f"No encontré las columnas esperadas. Encontré: {list(rows[0].keys())}"
        )

    cleaned = []
    seen = set()

    for r in rows:
        raw_id = str(r["chartmetric_id"]).strip()
        if not raw_id:
            continue
        try:
            cmid = int(float(raw_id))
        except ValueError:
            continue

        if cmid in seen:
            continue
        seen.add(cmid)

        cleaned.append({
            "chartmetric_id": cmid,
            "artist_name": str(r["artist_name"]).strip(),
            "chartmetric_url": str(r["chartmetric_url"]).strip()
        })

    return cleaned

def flatten_metadata(obj: dict, source_row: dict) -> dict:
    cm_stats = obj.get("cm_statistics", {}) or {}
    career_status = obj.get("career_status", {}) or {}
    genres = obj.get("genres", {}) or {}
    primary_genre = (genres.get("primary") or {}).get("name")


    return {
        "downloaded_at_utc": datetime.utcnow().isoformat(),
        "requested_chartmetric_id": source_row["chartmetric_id"],
        "requested_artist_name": source_row["artist_name"],
        "response_id": obj.get("id"),
        "name": obj.get("name"),
        "code2": obj.get("code2"),
        "pronoun_title": obj.get("pronoun_title"),
        "gender_title": obj.get("gender_title"),
        "record_label": obj.get("record_label"),
        "hometown_city": obj.get("hometown_city"),
        "cm_artist_rank": obj.get("cm_artist_rank"),
        "cm_artist_score": obj.get("cm_artist_score"),
        "career_stage": career_status.get("stage"),
        "career_stage_score": career_status.get("stage_score"),
        "career_trend": career_status.get("trend"),
        "career_trend_score": career_status.get("trend_score"),
        "primary_genre": primary_genre,
        "sp_followers": cm_stats.get("sp_followers"),
        "sp_monthly_listeners": cm_stats.get("sp_monthly_listeners"),
        "sp_popularity": cm_stats.get("sp_popularity"),
        "sp_playlist_total_reach": cm_stats.get("sp_playlist_total_reach"),
        "ins_followers": cm_stats.get("ins_followers"),
        "twitter_followers": cm_stats.get("twitter_followers"),
        "tiktok_followers": cm_stats.get("tiktok_followers"),
        "tiktok_likes": cm_stats.get("tiktok_likes"),
        "ycs_subscribers": cm_stats.get("ycs_subscribers"),
        "ycs_views": cm_stats.get("ycs_views"),
        "youtube_daily_video_views": cm_stats.get("youtube_daily_video_views"),
        "youtube_monthly_video_views": cm_stats.get("youtube_monthly_video_views"),
        "deezer_fans": cm_stats.get("deezer_fans"),
        "shazam_count": cm_stats.get("shazam_count"),
        "pandora_lifetime_streams": cm_stats.get("pandora_lifetime_streams"),
        "pandora_lifetime_stations_added": cm_stats.get("pandora_lifetime_stations_added")
    }

def write_csv_rows(path: Path, rows: list, fieldnames: list):
    file_exists = path.exists()
    with open(path, "a", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()
        for row in rows:
            writer.writerow(row)

# =========================
# READ INPUT
# =========================
artists = read_artist_file(INPUT_FILE)
pilot_artists = artists[:N_ARTISTS]

print(f"Total artistas leídos: {len(artists)}")
print(f"Piloto: {len(pilot_artists)} artistas")
print("Primeros 5:")
for a in pilot_artists[:5]:
    print(a)

# =========================
# AUTH
# =========================
access_token = get_access_token(REFRESH_TOKEN)
headers = build_headers(access_token)
print("\nAccess token obtenido.")

summary_rows = []
flat_rows = []

# =========================
# DOWNLOAD LOOP
# =========================
for idx, artist in enumerate(pilot_artists, start=1):
    artist_id = artist["chartmetric_id"]
    artist_name = artist["artist_name"]
    slug = safe_name(artist_name)

    url = f"https://api.chartmetric.com/api/artist/{artist_id}"

    print(f"\n[{idx}/{len(pilot_artists)}] {artist_name} ({artist_id})")

    try:
        time.sleep(SLEEP_SECONDS)

        resp = requests.get(url, headers=headers, timeout=90)

        # Si expiró el access token, renovamos y reintentamos 1 vez
        if resp.status_code == 401:
            print("401 -> renovando access token y reintentando...")
            access_token = get_access_token(REFRESH_TOKEN)
            headers = build_headers(access_token)
            time.sleep(SLEEP_SECONDS)
            resp = requests.get(url, headers=headers, timeout=90)

        status_code = resp.status_code

        rate_limit_limit = resp.headers.get("X-RateLimit-Limit")
        rate_limit_remaining = resp.headers.get("X-RateLimit-Remaining")
        rate_limit_reset = resp.headers.get("X-RateLimit-Reset")

        if status_code != 200:
            summary_rows.append({
                "downloaded_at_utc": datetime.utcnow().isoformat(),
                "chartmetric_id": artist_id,
                "artist_name": artist_name,
                "status_code": status_code,
                "ok": False,
                "error_text": resp.text[:500],
                "raw_json_path": "",
                "x_ratelimit_limit": rate_limit_limit,
                "x_ratelimit_remaining": rate_limit_remaining,
                "x_ratelimit_reset": rate_limit_reset
            })
            print(f"  status={status_code} error")
            continue

        data = resp.json()
        obj = data.get("obj", {})

        raw_path = RAW_DIR / f"artist_{artist_id}_{slug}_metadata.json"
        with open(raw_path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)

        summary_rows.append({
            "downloaded_at_utc": datetime.utcnow().isoformat(),
            "chartmetric_id": artist_id,
            "artist_name": artist_name,
            "status_code": status_code,
            "ok": True,
            "error_text": "",
            "raw_json_path": str(raw_path),
            "x_ratelimit_limit": rate_limit_limit,
            "x_ratelimit_remaining": rate_limit_remaining,
            "x_ratelimit_reset": rate_limit_reset
        })

        flat_rows.append(flatten_metadata(obj, artist))

        print(
            f"  OK | name={obj.get('name')} | "
            f"sp_monthly_listeners={(obj.get('cm_statistics') or {}).get('sp_monthly_listeners')} | "
            f"ins_followers={(obj.get('cm_statistics') or {}).get('ins_followers')}"
        )

    except Exception as e:
        summary_rows.append({
            "downloaded_at_utc": datetime.now(timezone.utc).isoformat(),
            "chartmetric_id": artist_id,
            "artist_name": artist_name,
            "status_code": "",
            "ok": False,
            "error_text": repr(e),
            "raw_json_path": "",
            "x_ratelimit_limit": "",
            "x_ratelimit_remaining": "",
            "x_ratelimit_reset": ""
        })
        print(f"  EXCEPTION: {e!r}")

# =========================
# SAVE OUTPUTS
# =========================
summary_fields = [
    "downloaded_at_utc", "chartmetric_id", "artist_name", "status_code", "ok",
    "error_text", "raw_json_path", "x_ratelimit_limit", "x_ratelimit_remaining", "x_ratelimit_reset"
]
write_csv_rows(SUMMARY_CSV, summary_rows, summary_fields)

if flat_rows:
    flat_fields = list(flat_rows[0].keys())
    write_csv_rows(FLAT_CSV, flat_rows, flat_fields)

print("\nListo.")
print(f"Resumen: {SUMMARY_CSV}")
print(f"Flat metadata: {FLAT_CSV}")
print(f"Raw JSON dir: {RAW_DIR}")
print(f"OK: {sum(1 for r in summary_rows if r['ok'])} / {len(summary_rows)}")

FileNotFoundError: [Errno 2] No such file or directory: 'df_1000_api.txt'

In [ ]:
print(SUMMARY_CSV)
print(FLAT_CSV)
sum(1 for r in summary_rows if r["ok"]), len(summary_rows)



chartmetric_metadata_pilot\download_summary.csv
chartmetric_metadata_pilot\metadata_flat.csv


(20, 20)

In [ ]:
flat_rows[0]

{'downloaded_at_utc': '2026-03-23T14:29:43.317838',
 'requested_chartmetric_id': 214945,
 'requested_artist_name': 'Bad Bunny',
 'response_id': 214945,
 'name': 'Bad Bunny',
 'code2': 'PR',
 'pronoun_title': 'he/him',
 'gender_title': 'male',
 'record_label': 'Rimas Entertainment',
 'hometown_city': 'Vega Baja',
 'cm_artist_rank': 4,
 'cm_artist_score': 98.97637727690513,
 'career_stage': 'superstar',
 'career_stage_score': 100,
 'career_trend': 'growth',
 'career_trend_score': 62,
 'primary_genre': 'reggaeton',
 'sp_followers': 112748463,
 'sp_monthly_listeners': 112734492,
 'sp_popularity': 100,
 'sp_playlist_total_reach': 810569843,
 'ins_followers': 54721804,
 'twitter_followers': 5046451,
 'tiktok_followers': 41700000,
 'tiktok_likes': 439100000,
 'ycs_subscribers': 52500000,
 'ycs_views': 46575719019,
 'youtube_daily_video_views': 471698,
 'youtube_monthly_video_views': 5145970,
 'deezer_fans': 1212,
 'shazam_count': 279025957,
 'pandora_lifetime_streams': 5151562758,
 'pandora_l

# DATOS

“El dataset base proviene de Make Music Equal (licencia CC BY-SA 4.0).
Este fue enriquecido con datos obtenidos mediante la API de Chartmetric, los cuales poseen restricciones de uso.
Por este motivo, las variables derivadas de dicha fuente fueron transformadas y agregadas de forma tal que no permiten reconstruir los valores originales.”

PASO 1: Base de artistas para obtener IDs antes de conectar a la API
Fuente: Make Music Equal (CC BY-SA 4.0)
La base de datos de artistas se descargó del dataset de acceso público MakeMusicEqual versión 3/3/2026: 936420, 10.
https://chartmetric-public.s3.us-west-2.amazonaws.com/make-music-equal/mme_artist_info.csv 
Se eliminaron registros duplicados [nombre + url de artista]: 172
Resultado: 936248 filas/artistas. 
Legal: This work is licensed under a Creative Commons Attribution-ShareAlike 4.0 International License.

PASO 2: Construcción de las cohortes de datos a extraer
Base de datos MakeMusicEqual ordenada por Chartmetric_rank (rank 1 a 936248)
-cohorte 1: 1400 registros 
-cohorte 2: 600 registros 

Notas:
Hay discontinuidades en el atributo Chartmetric_rank (faltan números en la secuencia) porque la base makemusicequal está actualizada al 3/3/26 aunque fueron descargados el 23/3/2026. Misma fecha de acceso a la API. Como la variable chartmetrick_rank es dinámica, esa diferencia de 20 días hace que un artista pueda tener rank = 5 el 3/3/2026 y rank = 9 el 23/3/26. Sin embargo, eso no invalida la selección del conjunto de artistas sobre la cual se construye la extracción de datos porque el objetivo fue definir un conjunto de artistas ordenados por popularidad (medida por una fuenta de datos confiable). Además la variable rank no será utilizada en el análisis. Y por último, se comprueba que el rank puede variar sutilmente pero el conjunto de artistas definidos es predominantemente coincidente. 

PASO 3: API. Extracción de datos
Límite 14 días, 100 request x día. (inicio 23/3/26)
Se apunta a dos endpoints: artist metadaga y artist live events
La extracción de datos se fragmenta por cohortes.


# COHORTE 1 0-1400

## codigo metadata API
incluye opción de actualizar el refresh token
trabaja sobre input file df_1000_api (construido desde dataset MME ordenado por chartmetric rank)

In [ ]:
#puedo redefinir input file para ampliar a futuro

INPUT_FILE = "df_1400_api.txt"
N_TO_DOWNLOAD = 200

In [ ]:
import os
import csv
import json
import time
from pathlib import Path
from datetime import datetime

import requests

# =========================
# CONFIG
# =========================
INPUT_FILE = "df_1400_api.txt"   # ajustá si hiciera falta
N_TO_DOWNLOAD = 4             # cantidad NUEVA a bajar hoy
SLEEP_SECONDS = 1.1

# podés dejar variable de entorno o pegarlo manualmente para hoy
REFRESH_TOKEN = "112E1dROGObsUx9hWKnrOxLQWrjDZ9YXwoByI9ynpN7QjtFY2Lq1E3uW7zvuIrpY"
# REFRESH_TOKEN = os.getenv("CHARTMETRIC_REFRESH_TOKEN")

if not REFRESH_TOKEN:
    raise ValueError("No encontré refresh token.")

BASE_DIR = Path("chartmetric_metadata")
RAW_DIR = BASE_DIR / "raw_json"
BASE_DIR.mkdir(exist_ok=True)
RAW_DIR.mkdir(exist_ok=True)

SUMMARY_CSV = BASE_DIR / "download_summary.csv"
FLAT_CSV = BASE_DIR / "metadata_flat.csv"

# =========================
# HELPERS
# =========================
def safe_name(name: str) -> str:
    return (
        str(name).strip().lower()
        .replace(" ", "_")
        .replace("/", "_")
        .replace("\\", "_")
        .replace(":", "_")
        .replace("?", "_")
        .replace("*", "_")
        .replace('"', "_")
        .replace("<", "_")
        .replace(">", "_")
        .replace("|", "_")
    )

def get_access_token(refresh_token: str) -> str:
    resp = requests.post(
        "https://api.chartmetric.com/api/token",
        json={"refreshtoken": refresh_token},
        timeout=60
    )
    resp.raise_for_status()
    data = resp.json()
    if "token" not in data:
        raise ValueError(f"No vino 'token' en la respuesta: {data}")
    return data["token"]

def build_headers(access_token: str) -> dict:
    return {
        "Authorization": f"Bearer {access_token}",
        "X-Accept-Partial-Data": "true"
    }

def read_artist_file(path: str):
    with open(path, "r", encoding="utf-8-sig", newline="") as f:
        sample = f.read(5000)
        f.seek(0)

        try:
            dialect = csv.Sniffer().sniff(sample, delimiters=",\t;|")
            delimiter = dialect.delimiter
        except csv.Error:
            delimiter = "\t"

        reader = csv.DictReader(f, delimiter=delimiter)
        rows = list(reader)

    required = {"chartmetric_id", "artist_name", "chartmetric_url"}
    if not rows:
        raise ValueError("El archivo no tiene filas legibles.")
    if not required.issubset(set(rows[0].keys())):
        raise ValueError(
            f"No encontré las columnas esperadas. Encontré: {list(rows[0].keys())}"
        )

    cleaned = []
    seen = set()

    for r in rows:
        raw_id = str(r["chartmetric_id"]).strip()
        if not raw_id:
            continue
        try:
            cmid = int(float(raw_id))
        except ValueError:
            continue

        if cmid in seen:
            continue
        seen.add(cmid)

        cleaned.append({
            "chartmetric_id": cmid,
            "artist_name": str(r["artist_name"]).strip(),
            "chartmetric_url": str(r["chartmetric_url"]).strip()
        })

    return cleaned

def read_existing_success_ids(summary_csv_path: Path):
    success_ids = set()

    if not summary_csv_path.exists():
        return success_ids

    with open(summary_csv_path, "r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            ok_value = str(row.get("ok", "")).strip().lower()
            cmid = str(row.get("chartmetric_id", "")).strip()

            if ok_value == "true" and cmid:
                try:
                    success_ids.add(int(float(cmid)))
                except ValueError:
                    pass

    return success_ids

def flatten_metadata(obj: dict, source_row: dict) -> dict:
    cm_stats = obj.get("cm_statistics", {}) or {}
    career_status = obj.get("career_status", {}) or {}
    genres = obj.get("genres", {}) or {}
    primary_genre = (genres.get("primary") or {}).get("name")

    return {
        "downloaded_at_utc": datetime.utcnow().isoformat(),
        "requested_chartmetric_id": source_row["chartmetric_id"],
        "requested_artist_name": source_row["artist_name"],
        "response_id": obj.get("id"),
        "name": obj.get("name"),
        "code2": obj.get("code2"),
        "pronoun_title": obj.get("pronoun_title"),
        "gender_title": obj.get("gender_title"),
        "record_label": obj.get("record_label"),
        "hometown_city": obj.get("hometown_city"),
        "cm_artist_rank": obj.get("cm_artist_rank"),
        "cm_artist_score": obj.get("cm_artist_score"),
        "career_stage": career_status.get("stage"),
        "career_stage_score": career_status.get("stage_score"),
        "career_trend": career_status.get("trend"),
        "career_trend_score": career_status.get("trend_score"),
        "primary_genre": primary_genre,
        "sp_followers": cm_stats.get("sp_followers"),
        "sp_monthly_listeners": cm_stats.get("sp_monthly_listeners"),
        "sp_popularity": cm_stats.get("sp_popularity"),
        "sp_playlist_total_reach": cm_stats.get("sp_playlist_total_reach"),
        "ins_followers": cm_stats.get("ins_followers"),
        "twitter_followers": cm_stats.get("twitter_followers"),
        "tiktok_followers": cm_stats.get("tiktok_followers"),
        "tiktok_likes": cm_stats.get("tiktok_likes"),
        "ycs_subscribers": cm_stats.get("ycs_subscribers"),
        "ycs_views": cm_stats.get("ycs_views"),
        "youtube_daily_video_views": cm_stats.get("youtube_daily_video_views"),
        "youtube_monthly_video_views": cm_stats.get("youtube_monthly_video_views"),
        "deezer_fans": cm_stats.get("deezer_fans"),
        "shazam_count": cm_stats.get("shazam_count"),
        "pandora_lifetime_streams": cm_stats.get("pandora_lifetime_streams"),
        "pandora_lifetime_stations_added": cm_stats.get("pandora_lifetime_stations_added")
    }

def write_csv_rows(path: Path, rows: list, fieldnames: list):
    if not rows:
        return

    file_exists = path.exists()
    with open(path, "a", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()
        for row in rows:
            writer.writerow(row)

# =========================
# READ INPUT + SKIP EXISTING
# =========================
artists = read_artist_file(INPUT_FILE)
already_downloaded_ids = read_existing_success_ids(SUMMARY_CSV)

pending_artists = [a for a in artists if a["chartmetric_id"] not in already_downloaded_ids]
batch_artists = pending_artists[:N_TO_DOWNLOAD]

print(f"Total artistas en archivo: {len(artists)}")
print(f"Ya descargados con ok=True: {len(already_downloaded_ids)}")
print(f"Pendientes: {len(pending_artists)}")
print(f"Se descargarán ahora: {len(batch_artists)}")

if len(batch_artists) == 0:
    print("No hay artistas nuevos para descargar.")
    raise SystemExit

print("\nPrimeros 10 del batch:")
for a in batch_artists[:10]:
    print(a)

# =========================
# AUTH CON REQUEST PARA TOKEN
# =========================
#access_token = get_access_token(REFRESH_TOKEN)
#headers = build_headers(access_token)
#print("\nAccess token obtenido.")

# =========================
# AUTH sin request para token
# =========================
# Reusamos el access_token ya obtenido en esta sesión.
# Solo renovaremos si aparece 401 durante la descarga.

if "access_token" not in globals():
    raise ValueError("No existe access_token en memoria. Si cerraste la sesión, habrá que pedir uno nuevo.")

if "headers" not in globals():
    headers = build_headers(access_token)

print("\nReusando access_token ya obtenido en esta sesión.")


summary_rows = []
flat_rows = []

# =========================
# DOWNLOAD LOOP
# =========================
for idx, artist in enumerate(batch_artists, start=1):
    artist_id = artist["chartmetric_id"]
    artist_name = artist["artist_name"]
    slug = safe_name(artist_name)

    url = f"https://api.chartmetric.com/api/artist/{artist_id}"

    print(f"\n[{idx}/{len(batch_artists)}] {artist_name} ({artist_id})")

    try:
        time.sleep(SLEEP_SECONDS)

        resp = requests.get(url, headers=headers, timeout=90)

        # renovar token si hiciera falta
        if resp.status_code == 401:
            print("401 -> renovando access token y reintentando...")
            access_token = get_access_token(REFRESH_TOKEN)
            headers = build_headers(access_token)
            time.sleep(SLEEP_SECONDS)
            resp = requests.get(url, headers=headers, timeout=90)

        status_code = resp.status_code

        rate_limit_limit = resp.headers.get("X-RateLimit-Limit")
        rate_limit_remaining = resp.headers.get("X-RateLimit-Remaining")
        rate_limit_reset = resp.headers.get("X-RateLimit-Reset")

        if status_code != 200:
            summary_rows.append({
                "downloaded_at_utc": datetime.utcnow().isoformat(),
                "chartmetric_id": artist_id,
                "artist_name": artist_name,
                "status_code": status_code,
                "ok": False,
                "error_text": resp.text[:500],
                "raw_json_path": "",
                "x_ratelimit_limit": rate_limit_limit,
                "x_ratelimit_remaining": rate_limit_remaining,
                "x_ratelimit_reset": rate_limit_reset
            })
            print(f"  status={status_code} error")
            print(f"  body={resp.text[:500]}")
            continue

        data = resp.json()
        obj = data.get("obj", {})

        raw_path = RAW_DIR / f"artist_{artist_id}_{slug}_metadata.json"
        with open(raw_path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)

        summary_rows.append({
            "downloaded_at_utc": datetime.utcnow().isoformat(),
            "chartmetric_id": artist_id,
            "artist_name": artist_name,
            "status_code": status_code,
            "ok": True,
            "error_text": "",
            "raw_json_path": str(raw_path),
            "x_ratelimit_limit": rate_limit_limit,
            "x_ratelimit_remaining": rate_limit_remaining,
            "x_ratelimit_reset": rate_limit_reset
        })

        flat_rows.append(flatten_metadata(obj, artist))

        print(
            f"  OK | name={obj.get('name')} | "
            f"sp_monthly_listeners={(obj.get('cm_statistics') or {}).get('sp_monthly_listeners')} | "
            f"remaining={rate_limit_remaining}"
        )

    except Exception as e:
        summary_rows.append({
            "downloaded_at_utc": datetime.utcnow().isoformat(),
            "chartmetric_id": artist_id,
            "artist_name": artist_name,
            "status_code": "",
            "ok": False,
            "error_text": repr(e),
            "raw_json_path": "",
            "x_ratelimit_limit": "",
            "x_ratelimit_remaining": "",
            "x_ratelimit_reset": ""
        })
        print(f"  EXCEPTION: {e!r}")

# =========================
# SAVE OUTPUTS
# =========================
summary_fields = [
    "downloaded_at_utc", "chartmetric_id", "artist_name", "status_code", "ok",
    "error_text", "raw_json_path", "x_ratelimit_limit", "x_ratelimit_remaining", "x_ratelimit_reset"
]
write_csv_rows(SUMMARY_CSV, summary_rows, summary_fields)

if flat_rows:
    flat_fields = list(flat_rows[0].keys())
    write_csv_rows(FLAT_CSV, flat_rows, flat_fields)

print("\nListo.")
print(f"Resumen: {SUMMARY_CSV}")
print(f"Flat metadata: {FLAT_CSV}")
print(f"Raw JSON dir: {RAW_DIR}")
print(f"Nuevos OK: {sum(1 for r in summary_rows if r['ok'])} / {len(summary_rows)}")
print(f"Total ya descargados acumulados (aprox): {len(already_downloaded_ids) + sum(1 for r in summary_rows if r['ok'])}")

Total artistas en archivo: 1400
Ya descargados con ok=True: 1399
Pendientes: 1
Se descargarán ahora: 1

Primeros 10 del batch:
{'chartmetric_id': 3917158, 'artist_name': 'David Kushner', 'chartmetric_url': 'https://app.chartmetric.com/artist/3917158'}

Reusando access_token ya obtenido en esta sesión.

[1/1] David Kushner (3917158)
  OK | name=David Kushner | sp_monthly_listeners=11779147 | remaining=510

Listo.
Resumen: chartmetric_metadata_pilot\download_summary.csv
Flat metadata: chartmetric_metadata_pilot\metadata_flat.csv
Raw JSON dir: chartmetric_metadata_pilot\raw_json
Nuevos OK: 1 / 1
Total ya descargados acumulados (aprox): 1400


C:\Users\Silvana\AppData\Local\Temp\ipykernel_24340\1023561803.py:290: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "downloaded_at_utc": datetime.utcnow().isoformat(),
C:\Users\Silvana\AppData\Local\Temp\ipykernel_24340\1023561803.py:141: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "downloaded_at_utc": datetime.utcnow().isoformat(),


## consolido metadata 1400

In [ ]:
#!pip install pandas numpy matplotlib seaborn scikit-learn jupyter

#!python -m pip install pandas


   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.7 MB 2.9 MB/s eta 0:00:04
   --- ------------------------------------ 0.8/9.7 MB 1.4 MB/s eta 0:00:07
   ---- ----------------------------------- 1.0/9.7 MB 1.7 MB/s eta 0:00:06
   ----- ---------------------------------- 1.3/9.7 MB 1.4 MB/s eta 0:00:07
   ------- -------------------------------- 1.8/9.7 MB 1.6 MB/s eta 0:00:05
   --------- ------------------------------ 2.4/9.7 MB 1.8 MB/s eta 0:00:05
   ------------ --------------------------- 3.1/9.7 MB 2.1 MB/s eta 0:00:04
   ------------- -------------------------- 3.4/9.7 MB 2.1 MB/s eta 0:00:04
   ----------------- ---------------------- 4.2/9.7 MB 2.1 MB/s eta 0:00:03
   -------------------- ------------------- 5.0/9.7 MB 2.3 MB/s eta 0:00:03
   ----------------------- ---------------- 5.8/9.7 MB 2.4 MB/s eta 0:00:02
   -----------------------


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd

# 1. Carga del archivo consolidado
ruta_metadata = "chartmetric_metadata/metadata_flat.csv"
df = pd.read_csv(ruta_metadata)

print(f"--- AUDITORÍA DE SALUD FINAL (BLOQUE 1400) ---")
print(f"Estructura: {df.shape[0]} artistas y {df.shape[1]} variables.")

# 2. Identificadores y Progreso
# Usamos el nombre exacto de tu columna: 'requested_chartmetric_id'
ids_unicos = df["requested_chartmetric_id"].nunique()
duplicados = df["requested_chartmetric_id"].duplicated().sum()

print(f"\n[IDENTIFICADORES]")
print(f"IDs únicos procesados: {ids_unicos}")
print(f"Registros duplicados:  {duplicados}")
print(f"Progreso de la meta:   {(ids_unicos / 1400) * 100:.2f}%")

# 3. Análisis de Calidad (NAs)
na_pct = (df.isna().mean() * 100).sort_values(ascending=False)

print("\n[CALIDAD DE DATA - % DE NULOS]")
# Mostramos las columnas con nulos (Top 15 para no saturar)
print(na_pct[na_pct > 0].head(15).round(2).astype(str) + '%')

# 4. Verificación del Ranking (Nombre corregido a 'cm_artist_rank')
col_rank = 'cm_artist_rank'
if col_rank in df.columns:
    rank_na = df[col_rank].isna().sum()
    print(f"\n✅ Columna '{col_rank}' detectada.")
    if rank_na > 0:
        print(f"⚠️ Hay {rank_na} artistas sin valor de rank en la metadata.")
    else:
        print(f"✅ Integridad total: 0 nulos en el ranking.")
else:
    print(f"\n❌ Error: No se encuentra la columna '{col_rank}'.")

print("-" * 50)

--- AUDITORÍA DE SALUD FINAL (BLOQUE 1400) ---
Estructura: 1400 artistas y 34 variables.

[IDENTIFICADORES]
IDs únicos procesados: 1400
Registros duplicados:  0
Progreso de la meta:   100.00%

[CALIDAD DE DATA - % DE NULOS]
gender_title                       30.36%
hometown_city                      22.93%
youtube_daily_video_views          19.29%
youtube_monthly_video_views        19.29%
tiktok_followers                   16.21%
tiktok_likes                       16.21%
twitter_followers                  13.57%
record_label                        6.14%
ins_followers                       3.71%
pandora_lifetime_stations_added     3.29%
ycs_views                           2.64%
pandora_lifetime_streams            2.64%
ycs_subscribers                     2.64%
deezer_fans                         0.93%
code2                               0.36%
dtype: str

✅ Columna 'cm_artist_rank' detectada.
✅ Integridad total: 0 nulos en el ranking.
--------------------------------------------------


## codigo live events 2025+2024 y mejoras

In [ ]:
import os
import csv
import json
import time
from pathlib import Path
from datetime import datetime, timezone

import requests

# =========================
# CONFIG
# =========================
METADATA_FILE = "chartmetric_metadata/metadata_flat.csv"
N_TO_DOWNLOAD = 1
SLEEP_SECONDS = 1.1

# Refresh token: se usa solo si no hay access_token en memoria
# o si durante el loop aparece 401
REFRESH_TOKEN = "112E1dROGObsUx9hWKnrOxLQWrjDZ9YXwoByI9ynpN7QjtFY2Lq1E3uW7zvuIrpY"

if not REFRESH_TOKEN:
    raise ValueError("No encontré refresh token.")

BASE_DIR = Path("chartmetric_liveevents_2024_2025")
RAW_DIR = BASE_DIR / "raw_json"
BASE_DIR.mkdir(exist_ok=True)
RAW_DIR.mkdir(exist_ok=True)

SUMMARY_CSV = BASE_DIR / "liveevents_2024_2025_summary.csv"
EVENTS_CSV = BASE_DIR / "liveevents_2024_2025_eventlevel.csv"

# Ventana: 2024-01-01 a 2025-12-31
# asumiendo hoy = 2026-03-24
FROM_DAYS_AGO = 813   # llega hasta 2024-01-01
TO_DAYS_AGO = 83      # corta en 2025-12-31
LIMIT = 100

# =========================
# HELPERS
# =========================
def now_utc_iso():
    return datetime.now(timezone.utc).isoformat()

def safe_name(name: str) -> str:
    return (
        str(name).strip().lower()
        .replace(" ", "_")
        .replace("/", "_")
        .replace("\\", "_")
        .replace(":", "_")
        .replace("?", "_")
        .replace("*", "_")
        .replace('"', "_")
        .replace("<", "_")
        .replace(">", "_")
        .replace("|", "_")
    )

def get_access_token(refresh_token: str) -> str:
    resp = requests.post(
        "https://api.chartmetric.com/api/token",
        json={"refreshtoken": refresh_token},
        timeout=60
    )
    resp.raise_for_status()
    data = resp.json()
    if "token" not in data:
        raise ValueError(f"No vino 'token' en la respuesta: {data}")
    return data["token"]

def build_headers(access_token: str) -> dict:
    return {
        "Authorization": f"Bearer {access_token}",
        "X-Accept-Partial-Data": "true"
    }

def write_csv_rows(path: Path, rows: list, fieldnames: list):
    if not rows:
        return

    file_exists = path.exists()
    with open(path, "a", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()
        for row in rows:
            writer.writerow(row)

def read_metadata_artists(path: str):
    artists = []
    seen = set()

    with open(path, "r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            raw_id = str(row.get("requested_chartmetric_id", "")).strip()
            artist_name = str(row.get("requested_artist_name", "")).strip()

            if not raw_id:
                continue

            try:
                cmid = int(float(raw_id))
            except ValueError:
                continue

            if cmid in seen:
                continue
            seen.add(cmid)

            artists.append({
                "chartmetric_id": cmid,
                "artist_name": artist_name
            })

    return artists

def read_existing_success_ids(summary_csv_path: Path):
    success_ids = set()

    if not summary_csv_path.exists():
        return success_ids

    with open(summary_csv_path, "r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            ok_value = str(row.get("ok", "")).strip().lower()
            cmid = str(row.get("chartmetric_id", "")).strip()

            if ok_value == "true" and cmid:
                try:
                    success_ids.add(int(float(cmid)))
                except ValueError:
                    pass

    return success_ids

def parse_event_year(start_date_value):
    """
    start_date viene como string tipo '2025-12-05 21:00:00.000'
    """
    if not start_date_value:
        return None

    s = str(start_date_value).strip()
    if len(s) < 4:
        return None

    try:
        return int(s[:4])
    except ValueError:
        return None

def flatten_event(event: dict, artist: dict):
    city_obj = event.get("city") or {}
    event_year = parse_event_year(event.get("start_date"))

    return {
        "downloaded_at_utc": now_utc_iso(),
        "chartmetric_id": artist["chartmetric_id"],
        "artist_name": artist["artist_name"],
        "event_id": event.get("id"),
        "event_name": event.get("event_name"),
        "type": event.get("type"),
        "event_year": event_year,
        "start_date": event.get("start_date"),
        "end_date": event.get("end_date"),
        "venue_id": event.get("venue_id"),
        "venue_name": event.get("venue_name"),
        "venue_capacity": event.get("venue_capacity"),
        "city_id": city_obj.get("id"),
        "city_name": city_obj.get("name"),
        "code2": event.get("code2"),
        "price": event.get("price"),
        "low_price": event.get("low_price"),
        "high_price": event.get("high_price"),
        "currency": event.get("currency"),
        "is_headliner": event.get("is_headliner"),
        "jambase_event_url": event.get("jambase_event_url"),
        "songkick_event_url": event.get("songkick_event_url"),
        "seatgeek_event_url": event.get("seatgeek_event_url"),
        "ticketmaster_event_url": event.get("ticketmaster_event_url")
    }

# =========================
# READ INPUT + SKIP EXISTING
# =========================
artists = read_metadata_artists(METADATA_FILE)
already_downloaded_ids = read_existing_success_ids(SUMMARY_CSV)

pending_artists = [a for a in artists if a["chartmetric_id"] not in already_downloaded_ids]
batch_artists = pending_artists[:N_TO_DOWNLOAD]

print(f"Total artistas en metadata: {len(artists)}")
print(f"Ya descargados liveevents ok=True: {len(already_downloaded_ids)}")
print(f"Pendientes: {len(pending_artists)}")
print(f"Se descargarán ahora: {len(batch_artists)}")

if len(batch_artists) == 0:
    print("No hay artistas nuevos para descargar.")
    raise SystemExit

# =========================
# AUTH
# =========================
# Si existe access_token en memoria, lo reutilizamos.
# Si no existe, pedimos uno nuevo una sola vez.

if "access_token" in globals():
    print("\nReusando access_token ya obtenido en esta sesión.")
    if "headers" not in globals():
        headers = build_headers(access_token)
else:
    print("\nNo había access_token en memoria. Pidiendo uno nuevo...")
    access_token = get_access_token(REFRESH_TOKEN)
    headers = build_headers(access_token)
    print("Access token obtenido.")

summary_rows = []
event_rows = []

# =========================
# DOWNLOAD LOOP
# =========================
for idx, artist in enumerate(batch_artists, start=1):
    artist_id = artist["chartmetric_id"]
    artist_name = artist["artist_name"]
    slug = safe_name(artist_name)

    print(f"\n[{idx}/{len(batch_artists)}] {artist_name} ({artist_id})")

    all_events = []
    offset = 0
    n_requests_for_artist = 0
    last_rate_limit_limit = None
    last_rate_limit_remaining = None
    last_rate_limit_reset = None
    finished_ok = False

    try:
        while True:
            time.sleep(SLEEP_SECONDS)

            url = f"https://api.chartmetric.com/api/artist/{artist_id}/past/events"
            params = {
                "fromDaysAgo": FROM_DAYS_AGO,
                "toDaysAgo": TO_DAYS_AGO,
                "limit": LIMIT,
                "offset": offset
            }

            resp = requests.get(url, headers=headers, params=params, timeout=90)

            # renovar token si hiciera falta
            if resp.status_code == 401:
                print("  401 -> renovando access token y reintentando...")
                access_token = get_access_token(REFRESH_TOKEN)
                headers = build_headers(access_token)
                time.sleep(SLEEP_SECONDS)
                resp = requests.get(url, headers=headers, params=params, timeout=90)

            n_requests_for_artist += 1

            last_rate_limit_limit = resp.headers.get("X-RateLimit-Limit")
            last_rate_limit_remaining = resp.headers.get("X-RateLimit-Remaining")
            last_rate_limit_reset = resp.headers.get("X-RateLimit-Reset")

            if resp.status_code != 200:
                summary_rows.append({
                    "downloaded_at_utc": now_utc_iso(),
                    "chartmetric_id": artist_id,
                    "artist_name": artist_name,
                    "ok": False,
                    "status_code": resp.status_code,
                    "error_text": resp.text[:500],
                    "raw_json_path": "",
                    "n_shows_2024": "",
                    "n_shows_2025": "",
                    "n_requests_live_2024_2025": n_requests_for_artist,
                    "x_ratelimit_limit": last_rate_limit_limit,
                    "x_ratelimit_remaining": last_rate_limit_remaining,
                    "x_ratelimit_reset": last_rate_limit_reset
                })
                print(f"  status={resp.status_code} error")
                break

            data = resp.json()
            page_events = data.get("obj", []) or []
            n_page = len(page_events)

            all_events.extend(page_events)

            print(
                f"  page offset={offset} | eventos={n_page} | "
                f"acumulados={len(all_events)} | remaining={last_rate_limit_remaining}"
            )

            # si vino menos que LIMIT, terminamos
            if n_page < LIMIT:
                n_shows_2024 = 0
                n_shows_2025 = 0

                for ev in all_events:
                    year = parse_event_year(ev.get("start_date"))
                    if year == 2024:
                        n_shows_2024 += 1
                    elif year == 2025:
                        n_shows_2025 += 1

                raw_payload = {
                    "chartmetric_id": artist_id,
                    "artist_name": artist_name,
                    "fromDaysAgo": FROM_DAYS_AGO,
                    "toDaysAgo": TO_DAYS_AGO,
                    "limit": LIMIT,
                    "n_requests_live_2024_2025": n_requests_for_artist,
                    "n_shows_2024": n_shows_2024,
                    "n_shows_2025": n_shows_2025,
                    "obj": all_events
                }

                raw_path = RAW_DIR / f"artist_{artist_id}_{slug}_liveevents_2024_2025.json"
                with open(raw_path, "w", encoding="utf-8") as f:
                    json.dump(raw_payload, f, ensure_ascii=False, indent=2)

                summary_rows.append({
                    "downloaded_at_utc": now_utc_iso(),
                    "chartmetric_id": artist_id,
                    "artist_name": artist_name,
                    "ok": True,
                    "status_code": 200,
                    "error_text": "",
                    "raw_json_path": str(raw_path),
                    "n_shows_2024": n_shows_2024,
                    "n_shows_2025": n_shows_2025,
                    "n_requests_live_2024_2025": n_requests_for_artist,
                    "x_ratelimit_limit": last_rate_limit_limit,
                    "x_ratelimit_remaining": last_rate_limit_remaining,
                    "x_ratelimit_reset": last_rate_limit_reset
                })

                for ev in all_events:
                    event_rows.append(flatten_event(ev, artist))

                print(
                    f"  OK FINAL | n_shows_2024={n_shows_2024} | "
                    f"n_shows_2025={n_shows_2025} | "
                    f"requests={n_requests_for_artist}"
                )
                finished_ok = True
                break

            offset += LIMIT

        # si no terminó ok y tampoco guardó error explícito, no hacemos nada extra
        # porque ya quedó registrado dentro del loop

    except Exception as e:
        summary_rows.append({
            "downloaded_at_utc": now_utc_iso(),
            "chartmetric_id": artist_id,
            "artist_name": artist_name,
            "ok": False,
            "status_code": "",
            "error_text": repr(e),
            "raw_json_path": "",
            "n_shows_2024": "",
            "n_shows_2025": "",
            "n_requests_live_2024_2025": n_requests_for_artist,
            "x_ratelimit_limit": last_rate_limit_limit,
            "x_ratelimit_remaining": last_rate_limit_remaining,
            "x_ratelimit_reset": last_rate_limit_reset
        })
        print(f"  EXCEPTION: {e!r}")

# =========================
# SAVE OUTPUTS
# =========================
summary_fields = [
    "downloaded_at_utc",
    "chartmetric_id",
    "artist_name",
    "ok",
    "status_code",
    "error_text",
    "raw_json_path",
    "n_shows_2024",
    "n_shows_2025",
    "n_requests_live_2024_2025",
    "x_ratelimit_limit",
    "x_ratelimit_remaining",
    "x_ratelimit_reset"
]
write_csv_rows(SUMMARY_CSV, summary_rows, summary_fields)

if event_rows:
    event_fields = list(event_rows[0].keys())
    write_csv_rows(EVENTS_CSV, event_rows, event_fields)

print("\nListo.")
print(f"Resumen: {SUMMARY_CSV}")
print(f"Event-level: {EVENTS_CSV}")
print(f"Raw JSON dir: {RAW_DIR}")
print(f"Nuevos OK: {sum(1 for r in summary_rows if r['ok'])} / {len(summary_rows)}")

Total artistas en metadata: 1400
Ya descargados liveevents ok=True: 1399
Pendientes: 1
Se descargarán ahora: 1

Reusando access_token ya obtenido en esta sesión.

[1/1] Chaka Khan (648)
  page offset=0 | eventos=41 | acumulados=41 | remaining=100
  OK FINAL | n_shows_2024=8 | n_shows_2025=33 | requests=1

Listo.
Resumen: chartmetric_liveevents_2024_2025\liveevents_2024_2025_summary.csv
Event-level: chartmetric_liveevents_2024_2025\liveevents_2024_2025_eventlevel.csv
Raw JSON dir: chartmetric_liveevents_2024_2025\raw_json
Nuevos OK: 1 / 1


## identifico filas duplicadas live

In [ ]:
import csv
from collections import defaultdict

SUMMARY_FILE = "chartmetric_liveevents_2024_2025/liveevents_2024_2025_summary.csv"

def to_int(value, default=0):
    try:
        return int(float(str(value).strip()))
    except:
        return default

# Leer filas
with open(SUMMARY_FILE, "r", encoding="utf-8-sig", newline="") as f:
    rows = list(csv.DictReader(f))

# Agrupar por chartmetric_id
rows_by_id = defaultdict(list)

for row in rows:
    cmid = to_int(row.get("chartmetric_id", ""))
    if cmid:
        rows_by_id[cmid].append(row)

# Detectar duplicados
duplicates = {k: v for k, v in rows_by_id.items() if len(v) > 1}

print("=== DUPLICADOS DETECTADOS ===")
print("Cantidad de artistas duplicados:", len(duplicates))
print()

# Mostrar detalle
for cmid, group in duplicates.items():
    print("======================================")
    print(f"chartmetric_id: {cmid}")
    print(f"repeticiones: {len(group)}")
    
    for i, row in enumerate(group, start=1):
        print(f"\n--- fila {i} ---")
        print("artist_name:", row.get("artist_name"))
        print("ok:", row.get("ok"))
        print("status_code:", row.get("status_code"))
        print("n_shows_2024:", row.get("n_shows_2024"))
        print("n_shows_2025:", row.get("n_shows_2025"))
        print("n_requests:", row.get("n_requests_live_2024_2025"))
        print("downloaded_at:", row.get("downloaded_at_utc"))
    
    print()

=== DUPLICADOS DETECTADOS ===
Cantidad de artistas duplicados: 1

chartmetric_id: 648
repeticiones: 2

--- fila 1 ---
artist_name: Chaka Khan
ok: False
status_code: 502
n_shows_2024: 
n_shows_2025: 
n_requests: 1
downloaded_at: 2026-03-25T15:14:46.591396+00:00

--- fila 2 ---
artist_name: Chaka Khan
ok: True
status_code: 200
n_shows_2024: 8
n_shows_2025: 33
n_requests: 1
downloaded_at: 2026-03-25T15:19:11.784224+00:00



## limpio filas duplicadas live

In [ ]:
import csv
from pathlib import Path

INPUT_SUMMARY = Path("chartmetric_liveevents_2024_2025/liveevents_2024_2025_summary.csv")
OUTPUT_SUMMARY = Path("chartmetric_liveevents_2024_2025/liveevents_2024_2025_summary_dedup.csv")

def to_int(value, default=0):
    try:
        return int(float(str(value).strip()))
    except:
        return default

# Leer todas las filas
with open(INPUT_SUMMARY, "r", encoding="utf-8-sig", newline="") as f:
    rows = list(csv.DictReader(f))

if not rows:
    raise ValueError("El summary está vacío.")

fieldnames = list(rows[0].keys())

# Quedarse con la última fila por chartmetric_id
latest_by_id = {}
for row in rows:
    cmid = to_int(row.get("chartmetric_id", ""))
    if cmid:
        latest_by_id[cmid] = row

dedup_rows = list(latest_by_id.values())

# Guardar archivo limpio
with open(OUTPUT_SUMMARY, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(dedup_rows)

# Reporte
unique_ids_before = set()
for row in rows:
    cmid = to_int(row.get("chartmetric_id", ""))
    if cmid:
        unique_ids_before.add(cmid)

unique_ids_after = set()
for row in dedup_rows:
    cmid = to_int(row.get("chartmetric_id", ""))
    if cmid:
        unique_ids_after.add(cmid)

print("=== LIMPIEZA SUMMARY ===")
print("Filas originales:", len(rows))
print("IDs únicos originales:", len(unique_ids_before))
print("Filas deduplicadas:", len(dedup_rows))
print("IDs únicos deduplicados:", len(unique_ids_after))
print("Duplicados eliminados:", len(rows) - len(dedup_rows))
print("Archivo nuevo:", OUTPUT_SUMMARY)

=== LIMPIEZA SUMMARY ===
Filas originales: 1401
IDs únicos originales: 1400
Filas deduplicadas: 1400
IDs únicos deduplicados: 1400
Duplicados eliminados: 1
Archivo nuevo: chartmetric_liveevents_2024_2025\liveevents_2024_2025_summary_dedup.csv


In [ ]:
import csv

CLEAN_FILE = "chartmetric_liveevents_2024_2025/liveevents_2024_2025_summary_dedup.csv"

with open(CLEAN_FILE, "r", encoding="utf-8-sig", newline="") as f:
    clean_rows = list(csv.DictReader(f))

ids = [int(float(r["chartmetric_id"])) for r in clean_rows if str(r.get("chartmetric_id", "")).strip()]
print("filas:", len(clean_rows))
print("ids únicos:", len(set(ids)))
print("duplicados:", len(ids) - len(set(ids)))

filas: 1400
ids únicos: 1400
duplicados: 0


## reasigno filenames

In [ ]:
import os
from pathlib import Path

BASE_DIR = Path("chartmetric_liveevents_2024_2025")

original = BASE_DIR / "liveevents_2024_2025_summary.csv"
dedup = BASE_DIR / "liveevents_2024_2025_summary_dedup.csv"
backup = BASE_DIR / "liveevents_2024_2025_summary_condup.csv"

# 1. ELIMINAR BACKUP PREVIO (Para evitar el FileExistsError)
if backup.exists():
    backup.unlink() # Esto borra el archivo 'condup' existente
    print(f"Archivo de backup previo eliminado para dejar espacio.")

# 2. Renombrar original → backup (Mueve el archivo con duplicados a la papelera de 'condup')
if original.exists():
    os.rename(original, backup)
    print("Original (con duplicados) renombrado a:", backup.name)
else:
    print("No se encontró el original. Quizás ya lo habías renombrado.")

# 3. Renombrar dedup → original (El archivo limpio pasa a ser el oficial)
if dedup.exists():
    os.rename(dedup, original)
    print("¡Éxito! El archivo DEDUP ahora es el oficial:", original.name)
else:
    print(f"No se encontró el archivo {dedup.name}. Revisa si el nombre es correcto.")

print("\nProceso terminado.")

Archivo de backup previo eliminado para dejar espacio.
Original (con duplicados) renombrado a: liveevents_2024_2025_summary_condup.csv
¡Éxito! El archivo DEDUP ahora es el oficial: liveevents_2024_2025_summary.csv

Proceso terminado.


# control cohorte 1

In [ ]:
import csv
from pathlib import Path

# =========================
# CONFIG
# =========================
METADATA_FILE = "chartmetric_metadata_pilot/metadata_flat.csv"
LIVE_SUMMARY_FILE = "chartmetric_liveevents_2024_2025/liveevents_2024_2025_summary.csv"


# =========================
# HELPERS
# =========================
def read_csv_rows(path):
    with open(path, "r", encoding="utf-8-sig", newline="") as f:
        return list(csv.DictReader(f))

def to_int(value, default=0):
    try:
        return int(float(str(value).strip()))
    except:
        return default

def to_bool_str(value):
    return str(value).strip().lower() == "true"

# =========================
# READ FILES
# =========================
metadata_rows = read_csv_rows(METADATA_FILE)
live_rows = read_csv_rows(LIVE_SUMMARY_FILE)

# =========================
# TOTAL COHORT FROM METADATA
# =========================
metadata_ids = []
seen_meta = set()

for row in metadata_rows:
    cmid = to_int(row.get("requested_chartmetric_id", ""))
    if cmid and cmid not in seen_meta:
        seen_meta.add(cmid)
        metadata_ids.append(cmid)

metadata_ids_set = set(metadata_ids)

# =========================
# LIVE EVENTS STATUS
# Si un artista aparece varias veces en el summary,
# nos quedamos con la última fila.
# =========================
live_by_id = {}

for row in live_rows:
    cmid = to_int(row.get("chartmetric_id", ""))
    if cmid:
        live_by_id[cmid] = row

live_ids_set = set(live_by_id.keys())

ok_ids = set()
error_ids = set()
pagination_ids = set()
pagination_ok_ids = set()
zero_show_ok_ids = set()
positive_show_ok_ids = set()

for cmid, row in live_by_id.items():
    ok = to_bool_str(row.get("ok", ""))
    n_requests = to_int(row.get("n_requests_live_2024_2025", "0"))
    n_2024 = to_int(row.get("n_shows_2024", "0"))
    n_2025 = to_int(row.get("n_shows_2025", "0"))
    total_shows = n_2024 + n_2025

    if ok:
        ok_ids.add(cmid)
        if total_shows == 0:
            zero_show_ok_ids.add(cmid)
        else:
            positive_show_ok_ids.add(cmid)
    else:
        error_ids.add(cmid)

    if n_requests > 1:
        pagination_ids.add(cmid)
        if ok:
            pagination_ok_ids.add(cmid)

pending_ids = metadata_ids_set - ok_ids
pending_because_error_ids = error_ids & pending_ids

# =========================
# PRINT MAIN STATUS
# =========================
print("=== STATUS GENERAL ===")
print("Total cohorte metadata:", len(metadata_ids_set))
print("Total filas live summary:", len(live_rows))
print("Artistas únicos vistos en live summary:", len(live_ids_set))
print("OK live events:", len(ok_ids))
print("Errores live events:", len(error_ids))
print("Pendientes live events:", len(pending_ids))

print("\n=== COBERTURA DE SHOWS ===")
print("OK con 0 shows (2024+2025):", len(zero_show_ok_ids))
print("OK con >0 shows (2024+2025):", len(positive_show_ok_ids))

print("\n=== PAGINACIÓN ===")
print("Artistas que requirieron más de 1 request:", len(pagination_ids))
print("De esos, resueltos OK:", len(pagination_ok_ids))
print("De esos, NO resueltos OK:", len(pagination_ids - pagination_ok_ids))

# =========================
# LISTAS ÚTILES PARA MAÑANA
# =========================
print("\n=== PENDIENTES (primeros 20) ===")
pending_preview = sorted(list(pending_ids))[:20]
print(pending_preview)

print("\n=== ERRORES / REINTENTAR (primeros 20) ===")
retry_preview = sorted(list(pending_because_error_ids))[:20]
print(retry_preview)

# =========================
# TOP ARTISTAS CON MÁS REQUESTS
# =========================
print("\n=== TOP 20 artistas por cantidad de requests live ===")
top_requests = []

for cmid, row in live_by_id.items():
    top_requests.append({
        "chartmetric_id": cmid,
        "artist_name": row.get("artist_name", ""),
        "ok": row.get("ok", ""),
        "n_requests_live_2024_2025": to_int(row.get("n_requests_live_2024_2025", "0")),
        "n_shows_2024": to_int(row.get("n_shows_2024", "0")),
        "n_shows_2025": to_int(row.get("n_shows_2025", "0")),
    })

top_requests = sorted(
    top_requests,
    key=lambda x: (
        x["n_requests_live_2024_2025"],
        x["n_shows_2024"] + x["n_shows_2025"]
    ),
    reverse=True
)

for row in top_requests[:20]:
    print(row)

# =========================
# OPCIONAL: ARTISTAS PENDIENTES CON NOMBRE
# =========================
meta_name_by_id = {}
for row in metadata_rows:
    cmid = to_int(row.get("requested_chartmetric_id", ""))
    if cmid and cmid not in meta_name_by_id:
        meta_name_by_id[cmid] = row.get("requested_artist_name", "")

print("\n=== PENDIENTES CON NOMBRE (primeros 20) ===")
for cmid in sorted(list(pending_ids))[:20]:
    print(cmid, "-", meta_name_by_id.get(cmid, ""))

=== STATUS GENERAL ===
Total cohorte metadata: 1400
Total filas live summary: 1400
Artistas únicos vistos en live summary: 1400
OK live events: 1400
Errores live events: 0
Pendientes live events: 0

=== COBERTURA DE SHOWS ===
OK con 0 shows (2024+2025): 396
OK con >0 shows (2024+2025): 1004

=== PAGINACIÓN ===
Artistas que requirieron más de 1 request: 46
De esos, resueltos OK: 46
De esos, NO resueltos OK: 0

=== PENDIENTES (primeros 20) ===
[]

=== ERRORES / REINTENTAR (primeros 20) ===
[]

=== TOP 20 artistas por cantidad de requests live ===
{'chartmetric_id': 63, 'artist_name': 'Air Supply', 'ok': 'True', 'n_requests_live_2024_2025': 2, 'n_shows_2024': 69, 'n_shows_2025': 110}
{'chartmetric_id': 285, 'artist_name': 'The Beach Boys', 'ok': 'True', 'n_requests_live_2024_2025': 2, 'n_shows_2024': 72, 'n_shows_2025': 92}
{'chartmetric_id': 489, 'artist_name': 'ZZ Top', 'ok': 'True', 'n_requests_live_2024_2025': 2, 'n_shows_2024': 60, 'n_shows_2025': 100}
{'chartmetric_id': 1481260, 'ar

# COHORTE 2 : 1401 a 7000

## metadata 1401_7000

In [31]:
import os
import csv
import json
import time
from pathlib import Path
from datetime import datetime
from datetime import datetime, UTC

import requests

# =========================
# CONFIG
# =========================
INPUT_FILE = "df_1401_7000_api.txt"   # ajustá si hiciera falta
N_TO_DOWNLOAD = 200
SLEEP_SECONDS = 1.1

# podés dejar variable de entorno o pegarlo manualmente para hoy
REFRESH_TOKEN = "Hl5UFQJi0UeqK2Ks56bWOxp7OHTUD2eNgIMbZQZUdTRlJUHgZrTH5SGiKulx10u9"
# gmail original "112E1dROGObsUx9hWKnrOxLQWrjDZ9YXwoByI9ynpN7QjtFY2Lq1E3uW7zvuIrpY"
# REFRESH_TOKEN = os.getenv("CHARTMETRIC_REFRESH_TOKEN")

if not REFRESH_TOKEN:
    raise ValueError("No encontré refresh token.")

BASE_DIR = Path("chartmetric_metadata_cohorte2")
RAW_DIR = BASE_DIR / "raw_json_cohorte2"
BASE_DIR.mkdir(exist_ok=True)
RAW_DIR.mkdir(exist_ok=True)

SUMMARY_CSV = BASE_DIR / "download_summary_cohorte2.csv"
FLAT_CSV = BASE_DIR / "metadata_flat_cohorte2.csv"

# =========================
# HELPERS
# =========================
def safe_name(name: str) -> str:
    return (
        str(name).strip().lower()
        .replace(" ", "_")
        .replace("/", "_")
        .replace("\\", "_")
        .replace(":", "_")
        .replace("?", "_")
        .replace("*", "_")
        .replace('"', "_")
        .replace("<", "_")
        .replace(">", "_")
        .replace("|", "_")
    )

def get_access_token(refresh_token: str) -> str:
    resp = requests.post(
        "https://api.chartmetric.com/api/token",
        json={"refreshtoken": refresh_token},
        timeout=60
    )
    resp.raise_for_status()
    data = resp.json()
    if "token" not in data:
        raise ValueError(f"No vino 'token' en la respuesta: {data}")
    return data["token"]

def build_headers(access_token: str) -> dict:
    return {
        "Authorization": f"Bearer {access_token}",
        "X-Accept-Partial-Data": "true"
    }

def read_artist_file(path: str):
    with open(path, "r", encoding="utf-8-sig", newline="") as f:
        sample = f.read(5000)
        f.seek(0)

        try:
            dialect = csv.Sniffer().sniff(sample, delimiters=",\t;|")
            delimiter = dialect.delimiter
        except csv.Error:
            delimiter = "\t"

        reader = csv.DictReader(f, delimiter=delimiter)
        rows = list(reader)

    required = {"chartmetric_id", "artist_name", "chartmetric_url"}
    if not rows:
        raise ValueError("El archivo no tiene filas legibles.")
    if not required.issubset(set(rows[0].keys())):
        raise ValueError(
            f"No encontré las columnas esperadas. Encontré: {list(rows[0].keys())}"
        )

    cleaned = []
    seen = set()

    for r in rows:
        raw_id = str(r["chartmetric_id"]).strip()
        if not raw_id:
            continue
        try:
            cmid = int(float(raw_id))
        except ValueError:
            continue

        if cmid in seen:
            continue
        seen.add(cmid)

        cleaned.append({
            "chartmetric_id": cmid,
            "artist_name": str(r["artist_name"]).strip(),
            "chartmetric_url": str(r["chartmetric_url"]).strip()
        })

    return cleaned

def read_existing_success_ids(summary_csv_path: Path):
    success_ids = set()

    if not summary_csv_path.exists():
        return success_ids

    with open(summary_csv_path, "r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            ok_value = str(row.get("ok", "")).strip().lower()
            cmid = str(row.get("chartmetric_id", "")).strip()

            if ok_value == "true" and cmid:
                try:
                    success_ids.add(int(float(cmid)))
                except ValueError:
                    pass

    return success_ids

def flatten_metadata(obj: dict, source_row: dict) -> dict:
    cm_stats = obj.get("cm_statistics", {}) or {}
    career_status = obj.get("career_status", {}) or {}
    genres = obj.get("genres", {}) or {}
    primary_genre = (genres.get("primary") or {}).get("name")

    return {
        "downloaded_at_utc": datetime.utcnow().isoformat(),
        "requested_chartmetric_id": source_row["chartmetric_id"],
        "requested_artist_name": source_row["artist_name"],
        "response_id": obj.get("id"),
        "name": obj.get("name"),
        "code2": obj.get("code2"),
        "pronoun_title": obj.get("pronoun_title"),
        "gender_title": obj.get("gender_title"),
        "record_label": obj.get("record_label"),
        "hometown_city": obj.get("hometown_city"),
        "cm_artist_rank": obj.get("cm_artist_rank"),
        "cm_artist_score": obj.get("cm_artist_score"),
        "career_stage": career_status.get("stage"),
        "career_stage_score": career_status.get("stage_score"),
        "career_trend": career_status.get("trend"),
        "career_trend_score": career_status.get("trend_score"),
        "primary_genre": primary_genre,
        "sp_followers": cm_stats.get("sp_followers"),
        "sp_monthly_listeners": cm_stats.get("sp_monthly_listeners"),
        "sp_popularity": cm_stats.get("sp_popularity"),
        "sp_playlist_total_reach": cm_stats.get("sp_playlist_total_reach"),
        "ins_followers": cm_stats.get("ins_followers"),
        "twitter_followers": cm_stats.get("twitter_followers"),
        "tiktok_followers": cm_stats.get("tiktok_followers"),
        "tiktok_likes": cm_stats.get("tiktok_likes"),
        "ycs_subscribers": cm_stats.get("ycs_subscribers"),
        "ycs_views": cm_stats.get("ycs_views"),
        "youtube_daily_video_views": cm_stats.get("youtube_daily_video_views"),
        "youtube_monthly_video_views": cm_stats.get("youtube_monthly_video_views"),
        "deezer_fans": cm_stats.get("deezer_fans"),
        "shazam_count": cm_stats.get("shazam_count"),
        "pandora_lifetime_streams": cm_stats.get("pandora_lifetime_streams"),
        "pandora_lifetime_stations_added": cm_stats.get("pandora_lifetime_stations_added")
    }

def write_csv_rows(path: Path, rows: list, fieldnames: list):
    if not rows:
        return

    file_exists = path.exists()
    with open(path, "a", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()
        for row in rows:
            writer.writerow(row)

# =========================
# READ INPUT + SKIP EXISTING
# =========================
artists = read_artist_file(INPUT_FILE)
already_downloaded_ids = read_existing_success_ids(SUMMARY_CSV)

pending_artists = [a for a in artists if a["chartmetric_id"] not in already_downloaded_ids]
batch_artists = pending_artists[:N_TO_DOWNLOAD]

print(f"Total artistas en archivo: {len(artists)}")
print(f"Ya descargados con ok=True: {len(already_downloaded_ids)}")
print(f"Pendientes: {len(pending_artists)}")
print(f"Se descargarán ahora: {len(batch_artists)}")

if len(batch_artists) == 0:
    print("No hay artistas nuevos para descargar.")
    raise SystemExit

print("\nPrimeros 10 del batch:")
for a in batch_artists[:10]:
    print(a)

# =========================
# AUTH CON REQUEST PARA TOKEN
# =========================
#access_token = get_access_token(REFRESH_TOKEN)
#headers = build_headers(access_token)
#print("\nAccess token obtenido.")

# =========================
# AUTH sin request para token
# =========================
# Reusamos el access_token ya obtenido en esta sesión.
# Solo renovaremos si aparece 401 durante la descarga.

# if "access_token" not in globals():
#     raise ValueError("No existe access_token en memoria. Si cerraste la sesión, habrá que pedir uno nuevo.")

# if "headers" not in globals():
#     headers = build_headers(access_token)

# print("\nReusando access_token ya obtenido en esta sesión.")

access_token = get_access_token(REFRESH_TOKEN)
headers = build_headers(access_token)
print(headers)
print("\nAccess token obtenido.")

summary_rows = []
flat_rows = []

# =========================
# DOWNLOAD LOOP
# =========================
for idx, artist in enumerate(batch_artists, start=1):
    artist_id = artist["chartmetric_id"]
    artist_name = artist["artist_name"]
    slug = safe_name(artist_name)

    url = f"https://api.chartmetric.com/api/artist/{artist_id}"

    print(f"\n[{idx}/{len(batch_artists)}] {artist_name} ({artist_id})")

    try:
        time.sleep(SLEEP_SECONDS)

        resp = requests.get(url, headers=headers, timeout=90)

        # renovar token si hiciera falta
        if resp.status_code == 401:
            print("401 -> renovando access token y reintentando...")
            access_token = get_access_token(REFRESH_TOKEN)
            headers = build_headers(access_token)
            time.sleep(SLEEP_SECONDS)
            resp = requests.get(url, headers=headers, timeout=90)

        status_code = resp.status_code

        rate_limit_limit = resp.headers.get("X-RateLimit-Limit")
        rate_limit_remaining = resp.headers.get("X-RateLimit-Remaining")
        rate_limit_reset = resp.headers.get("X-RateLimit-Reset")

        if status_code != 200:
            summary_rows.append({
                "downloaded_at_utc": datetime.utcnow().isoformat(),
                "chartmetric_id": artist_id,
                "artist_name": artist_name,
                "status_code": status_code,
                "ok": False,
                "error_text": resp.text[:500],
                "raw_json_path": "",
                "x_ratelimit_limit": rate_limit_limit,
                "x_ratelimit_remaining": rate_limit_remaining,
                "x_ratelimit_reset": rate_limit_reset
            })
            print(f"  status={status_code} error")
            continue

        data = resp.json()
        obj = data.get("obj", {})

        raw_path = RAW_DIR / f"artist_{artist_id}_{slug}_metadata.json"
        with open(raw_path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)

        summary_rows.append({
            "downloaded_at_utc": datetime.utcnow().isoformat(),
            "chartmetric_id": artist_id,
            "artist_name": artist_name,
            "status_code": status_code,
            "ok": True,
            "error_text": "",
            "raw_json_path": str(raw_path),
            "x_ratelimit_limit": rate_limit_limit,
            "x_ratelimit_remaining": rate_limit_remaining,
            "x_ratelimit_reset": rate_limit_reset
        })

        flat_rows.append(flatten_metadata(obj, artist))

        print(
            f"  OK | name={obj.get('name')} | "
            f"sp_monthly_listeners={(obj.get('cm_statistics') or {}).get('sp_monthly_listeners')} | "
            f"remaining={rate_limit_remaining}"
        )

    except Exception as e:
        summary_rows.append({
            "downloaded_at_utc": datetime.now(UTC).isoformat(),
            "chartmetric_id": artist_id,
            "artist_name": artist_name,
            "status_code": "",
            "ok": False,
            "error_text": repr(e),
            "raw_json_path": "",
            "x_ratelimit_limit": "",
            "x_ratelimit_remaining": "",
            "x_ratelimit_reset": ""
        })
        print(f"  EXCEPTION: {e!r}")

# =========================
# SAVE OUTPUTS
# =========================
summary_fields = [
    "downloaded_at_utc", "chartmetric_id", "artist_name", "status_code", "ok",
    "error_text", "raw_json_path", "x_ratelimit_limit", "x_ratelimit_remaining", "x_ratelimit_reset"
]
write_csv_rows(SUMMARY_CSV, summary_rows, summary_fields)

if flat_rows:
    flat_fields = list(flat_rows[0].keys())
    write_csv_rows(FLAT_CSV, flat_rows, flat_fields)

print("\nListo.")
print(f"Resumen: {SUMMARY_CSV}")
print(f"Flat metadata: {FLAT_CSV}")
print(f"Raw JSON dir: {RAW_DIR}")
print(f"Nuevos OK: {sum(1 for r in summary_rows if r['ok'])} / {len(summary_rows)}")
print(f"Total ya descargados acumulados (aprox): {len(already_downloaded_ids) + sum(1 for r in summary_rows if r['ok'])}")

Total artistas en archivo: 5600
Ya descargados con ok=True: 5400
Pendientes: 200
Se descargarán ahora: 200

Primeros 10 del batch:
{'chartmetric_id': 424548, 'artist_name': 'Kathryn Stott', 'chartmetric_url': 'https://app.chartmetric.com/artist/424548'}
{'chartmetric_id': 4489475, 'artist_name': 'Real Ona', 'chartmetric_url': 'https://app.chartmetric.com/artist/4489475'}
{'chartmetric_id': 2536, 'artist_name': 'Marsha Ambrosius', 'chartmetric_url': 'https://app.chartmetric.com/artist/2536'}
{'chartmetric_id': 413516, 'artist_name': 'JIGGO', 'chartmetric_url': 'https://app.chartmetric.com/artist/413516'}
{'chartmetric_id': 3719106, 'artist_name': 'Akshath', 'chartmetric_url': 'https://app.chartmetric.com/artist/3719106'}
{'chartmetric_id': 3027, 'artist_name': 'Laura Marling', 'chartmetric_url': 'https://app.chartmetric.com/artist/3027'}
{'chartmetric_id': 214284, 'artist_name': 'Rachel House', 'chartmetric_url': 'https://app.chartmetric.com/artist/214284'}
{'chartmetric_id': 19206, 'ar

C:\Users\Silvana\AppData\Local\Temp\ipykernel_8400\1484626313.py:296: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "downloaded_at_utc": datetime.utcnow().isoformat(),
C:\Users\Silvana\AppData\Local\Temp\ipykernel_8400\1484626313.py:143: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "downloaded_at_utc": datetime.utcnow().isoformat(),


  OK | name=Kathryn Stott | sp_monthly_listeners=3093349 | remaining=24

[2/200] Real Ona (4489475)
  OK | name=Real Ona | sp_monthly_listeners=100 | remaining=24

[3/200] Marsha Ambrosius (2536)
  OK | name=Marsha Ambrosius | sp_monthly_listeners=276955 | remaining=24

[4/200] JIGGO (413516)
  OK | name=JIGGO | sp_monthly_listeners=283565 | remaining=24

[5/200] Akshath (3719106)
  OK | name=Akshath | sp_monthly_listeners=1495926 | remaining=24

[6/200] Laura Marling (3027)
  OK | name=Laura Marling | sp_monthly_listeners=923374 | remaining=24

[7/200] Rachel House (214284)
  OK | name=Rachel House | sp_monthly_listeners=2730673 | remaining=24

[8/200] Las Pastillas del Abuelo (19206)
  OK | name=Las Pastillas del Abuelo | sp_monthly_listeners=1525283 | remaining=24

[9/200] Chace (475292)
  OK | name=Chace | sp_monthly_listeners=131595 | remaining=24

[10/200] Dario Marianelli (1877)
  OK | name=Dario Marianelli | sp_monthly_listeners=1658858 | remaining=24

[11/200] Cafe Music BGM c

## consolido metadata cohorte2

In [32]:
import pandas as pd

# 1. Carga del archivo consolidado
ruta_metadata = "chartmetric_metadata_cohorte2/metadata_flat_cohorte2.csv"
df2 = pd.read_csv(ruta_metadata)

print(f"--- AUDITORÍA DE SALUD FINAL (BLOQUE 1400) ---")
print(f"Estructura: {df2.shape[0]} artistas y {df2.shape[1]} variables.")

# 2. Identificadores y Progreso
# Usamos el nombre exacto de tu columna: 'requested_chartmetric_id'
ids_unicos = df2["requested_chartmetric_id"].nunique()
duplicados = df2["requested_chartmetric_id"].duplicated().sum()

print(f"\n[IDENTIFICADORES]")
print(f"IDs únicos procesados: {ids_unicos}")
print(f"Registros duplicados:  {duplicados}")
print(f"Progreso de la meta:   {(ids_unicos / 1400) * 100:.2f}%")

# 3. Análisis de Calidad (NAs)
na_pct = (df2.isna().mean() * 100).sort_values(ascending=False)

print("\n[CALIDAD DE DATA - % DE NULOS]")
# Mostramos las columnas con nulos (Top 15 para no saturar)
print(na_pct[na_pct > 0].head(15).round(2).astype(str) + '%')

# 4. Verificación del Ranking (Nombre corregido a 'cm_artist_rank')
col_rank = 'cm_artist_rank'
if col_rank in df2.columns:
    rank_na = df2[col_rank].isna().sum()
    print(f"\n✅ Columna '{col_rank}' detectada.")
    if rank_na > 0:
        print(f"⚠️ Hay {rank_na} artistas sin valor de rank en la metadata.")
    else:
        print(f"✅ Integridad total: 0 nulos en el ranking.")
else:
    print(f"\n❌ Error: No se encuentra la columna '{col_rank}'.")

print("-" * 50)

--- AUDITORÍA DE SALUD FINAL (BLOQUE 1400) ---
Estructura: 5600 artistas y 34 variables.

[IDENTIFICADORES]
IDs únicos procesados: 5600
Registros duplicados:  0
Progreso de la meta:   400.00%

[CALIDAD DE DATA - % DE NULOS]
gender_title                       73.86%
youtube_monthly_video_views        43.68%
youtube_daily_video_views          43.68%
hometown_city                      40.84%
tiktok_followers                   35.21%
tiktok_likes                       35.21%
twitter_followers                  28.27%
band                               19.96%
pandora_lifetime_stations_added    11.14%
ycs_subscribers                    10.39%
ycs_views                          10.39%
ins_followers                       9.89%
record_label                        7.82%
pandora_lifetime_streams            7.73%
deezer_fans                          3.3%
dtype: str

✅ Columna 'cm_artist_rank' detectada.
✅ Integridad total: 0 nulos en el ranking.
--------------------------------------------------


## codigo live events 2025+2024 y mejoras

In [36]:
import os
import csv
import json
import time
from pathlib import Path
from datetime import datetime, timezone

import requests

# =========================
# CONFIG
# =========================
METADATA_FILE = "chartmetric_metadata_cohorte2/metadata_flat_cohorte2.csv"
N_TO_DOWNLOAD = 1

SLEEP_SECONDS = 1.1

# Refresh token: se usa solo si no hay access_token en memoria
# o si durante el loop aparece 401

#REFRESH_TOKEN = "Hl5UFQJi0UeqK2Ks56bWOxp7OHTUD2eNgIMbZQZUdTRlJUHgZrTH5SGiKulx10u9" #brave 6-4-25

REFRESH_TOKEN = "112E1dROGObsUx9hWKnrOxLQWrjDZ9YXwoByI9ynpN7QjtFY2Lq1E3uW7zvuIrpY" # gmail original 23-3-26

if not REFRESH_TOKEN:
    raise ValueError("No encontré refresh token.")

BASE_DIR = Path("chartmetric_liveevents_2024_2025_cohorte2")
RAW_DIR = BASE_DIR / "raw_json_cohorte2"
BASE_DIR.mkdir(exist_ok=True)
RAW_DIR.mkdir(exist_ok=True)

SUMMARY_CSV = BASE_DIR / "liveevents_2024_2025_summary_cohorte2.csv"
EVENTS_CSV = BASE_DIR / "liveevents_2024_2025_eventlevel_cohorte2.csv"

# Ventana: 2024-01-01 a 2025-12-31
# asumiendo hoy = 2026-04-5
FROM_DAYS_AGO = 825   # llega hasta 2024-01-01
TO_DAYS_AGO = 95      # corta en 2025-12-31
LIMIT = 100

# =========================
# HELPERS
# =========================
def now_utc_iso():
    return datetime.now(timezone.utc).isoformat()

def safe_name(name: str) -> str:
    return (
        str(name).strip().lower()
        .replace(" ", "_")
        .replace("/", "_")
        .replace("\\", "_")
        .replace(":", "_")
        .replace("?", "_")
        .replace("*", "_")
        .replace('"', "_")
        .replace("<", "_")
        .replace(">", "_")
        .replace("|", "_")
    )

def get_access_token(refresh_token: str) -> str:
    resp = requests.post(
        "https://api.chartmetric.com/api/token",
        json={"refreshtoken": refresh_token},
        timeout=60
    )
    resp.raise_for_status()
    data = resp.json()
    if "token" not in data:
        raise ValueError(f"No vino 'token' en la respuesta: {data}")
    return data["token"]

def build_headers(access_token: str) -> dict:
    return {
        "Authorization": f"Bearer {access_token}",
        "X-Accept-Partial-Data": "true"
    }

def write_csv_rows(path: Path, rows: list, fieldnames: list):
    if not rows:
        return

    file_exists = path.exists()
    with open(path, "a", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()
        for row in rows:
            writer.writerow(row)

def read_metadata_artists(path: str):
    artists = []
    seen = set()

    with open(path, "r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            raw_id = str(row.get("requested_chartmetric_id", "")).strip()
            artist_name = str(row.get("requested_artist_name", "")).strip()

            if not raw_id:
                continue

            try:
                cmid = int(float(raw_id))
            except ValueError:
                continue

            if cmid in seen:
                continue
            seen.add(cmid)

            artists.append({
                "chartmetric_id": cmid,
                "artist_name": artist_name
            })

    return artists

def read_existing_success_ids(summary_csv_path: Path):
    success_ids = set()

    if not summary_csv_path.exists():
        return success_ids

    with open(summary_csv_path, "r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            ok_value = str(row.get("ok", "")).strip().lower()
            cmid = str(row.get("chartmetric_id", "")).strip()

            if ok_value == "true" and cmid:
                try:
                    success_ids.add(int(float(cmid)))
                except ValueError:
                    pass

    return success_ids

def parse_event_year(start_date_value):
    """
    start_date viene como string tipo '2025-12-05 21:00:00.000'
    """
    if not start_date_value:
        return None

    s = str(start_date_value).strip()
    if len(s) < 4:
        return None

    try:
        return int(s[:4])
    except ValueError:
        return None

def flatten_event(event: dict, artist: dict):
    city_obj = event.get("city") or {}
    event_year = parse_event_year(event.get("start_date"))

    return {
        "downloaded_at_utc": now_utc_iso(),
        "chartmetric_id": artist["chartmetric_id"],
        "artist_name": artist["artist_name"],
        "event_id": event.get("id"),
        "event_name": event.get("event_name"),
        "type": event.get("type"),
        "event_year": event_year,
        "start_date": event.get("start_date"),
        "end_date": event.get("end_date"),
        "venue_id": event.get("venue_id"),
        "venue_name": event.get("venue_name"),
        "venue_capacity": event.get("venue_capacity"),
        "city_id": city_obj.get("id"),
        "city_name": city_obj.get("name"),
        "code2": event.get("code2"),
        "price": event.get("price"),
        "low_price": event.get("low_price"),
        "high_price": event.get("high_price"),
        "currency": event.get("currency"),
        "is_headliner": event.get("is_headliner"),
        "jambase_event_url": event.get("jambase_event_url"),
        "songkick_event_url": event.get("songkick_event_url"),
        "seatgeek_event_url": event.get("seatgeek_event_url"),
        "ticketmaster_event_url": event.get("ticketmaster_event_url")
    }

# =========================
# READ INPUT + SKIP EXISTING
# =========================
artists = read_metadata_artists(METADATA_FILE)
already_downloaded_ids = read_existing_success_ids(SUMMARY_CSV)

pending_artists = [a for a in artists if a["chartmetric_id"] not in already_downloaded_ids]
batch_artists = pending_artists[:N_TO_DOWNLOAD]

print(f"Total artistas en metadata: {len(artists)}")
print(f"Ya descargados liveevents ok=True: {len(already_downloaded_ids)}")
print(f"Pendientes: {len(pending_artists)}")
print(f"Se descargarán ahora: {len(batch_artists)}")

if len(batch_artists) == 0:
    print("No hay artistas nuevos para descargar.")
    raise SystemExit

# =========================
# AUTH
# =========================
# Si existe access_token en memoria, lo reutilizamos.
# Si no existe, pedimos uno nuevo una sola vez.

if "access_token" in globals():
    print("\nReusando access_token ya obtenido en esta sesión.")
    if "headers" not in globals():
        headers = build_headers(access_token)
else:
    print("\nNo había access_token en memoria. Pidiendo uno nuevo...")
    access_token = get_access_token(REFRESH_TOKEN)
    headers = build_headers(access_token)
    print("Access token obtenido.")

summary_rows = []
event_rows = []

# =========================
# DOWNLOAD LOOP
# =========================
for idx, artist in enumerate(batch_artists, start=1):
    artist_id = artist["chartmetric_id"]
    artist_name = artist["artist_name"]
    slug = safe_name(artist_name)

    print(f"\n[{idx}/{len(batch_artists)}] {artist_name} ({artist_id})")

    all_events = []
    offset = 0
    n_requests_for_artist = 0
    last_rate_limit_limit = None
    last_rate_limit_remaining = None
    last_rate_limit_reset = None
    finished_ok = False

    try:
        while True:
            time.sleep(SLEEP_SECONDS)

            url = f"https://api.chartmetric.com/api/artist/{artist_id}/past/events"
            params = {
                "fromDaysAgo": FROM_DAYS_AGO,
                "toDaysAgo": TO_DAYS_AGO,
                "limit": LIMIT,
                "offset": offset
            }

            resp = requests.get(url, headers=headers, params=params, timeout=90)

            # renovar token si hiciera falta
            if resp.status_code == 401:
                print("  401 -> renovando access token y reintentando...")
                access_token = get_access_token(REFRESH_TOKEN)
                headers = build_headers(access_token)
                time.sleep(SLEEP_SECONDS)
                resp = requests.get(url, headers=headers, params=params, timeout=90)

            n_requests_for_artist += 1

            last_rate_limit_limit = resp.headers.get("X-RateLimit-Limit")
            last_rate_limit_remaining = resp.headers.get("X-RateLimit-Remaining")
            last_rate_limit_reset = resp.headers.get("X-RateLimit-Reset")

            if resp.status_code != 200:
                summary_rows.append({
                    "downloaded_at_utc": now_utc_iso(),
                    "chartmetric_id": artist_id,
                    "artist_name": artist_name,
                    "ok": False,
                    "status_code": resp.status_code,
                    "error_text": resp.text[:500],
                    "raw_json_path": "",
                    "n_shows_2024": "",
                    "n_shows_2025": "",
                    "n_requests_live_2024_2025": n_requests_for_artist,
                    "x_ratelimit_limit": last_rate_limit_limit,
                    "x_ratelimit_remaining": last_rate_limit_remaining,
                    "x_ratelimit_reset": last_rate_limit_reset
                })
                print(f"  status={resp.status_code} error")
                break

            data = resp.json()
            page_events = data.get("obj", []) or []
            n_page = len(page_events)

            all_events.extend(page_events)

            print(
                f"  page offset={offset} | eventos={n_page} | "
                f"acumulados={len(all_events)} | remaining={last_rate_limit_remaining}"
            )

            # si vino menos que LIMIT, terminamos
            if n_page < LIMIT:
                n_shows_2024 = 0
                n_shows_2025 = 0

                for ev in all_events:
                    year = parse_event_year(ev.get("start_date"))
                    if year == 2024:
                        n_shows_2024 += 1
                    elif year == 2025:
                        n_shows_2025 += 1

                raw_payload = {
                    "chartmetric_id": artist_id,
                    "artist_name": artist_name,
                    "fromDaysAgo": FROM_DAYS_AGO,
                    "toDaysAgo": TO_DAYS_AGO,
                    "limit": LIMIT,
                    "n_requests_live_2024_2025": n_requests_for_artist,
                    "n_shows_2024": n_shows_2024,
                    "n_shows_2025": n_shows_2025,
                    "obj": all_events
                }

                raw_path = RAW_DIR / f"artist_{artist_id}_{slug}_liveevents_2024_2025.json"
                with open(raw_path, "w", encoding="utf-8") as f:
                    json.dump(raw_payload, f, ensure_ascii=False, indent=2)

                summary_rows.append({
                    "downloaded_at_utc": now_utc_iso(),
                    "chartmetric_id": artist_id,
                    "artist_name": artist_name,
                    "ok": True,
                    "status_code": 200,
                    "error_text": "",
                    "raw_json_path": str(raw_path),
                    "n_shows_2024": n_shows_2024,
                    "n_shows_2025": n_shows_2025,
                    "n_requests_live_2024_2025": n_requests_for_artist,
                    "x_ratelimit_limit": last_rate_limit_limit,
                    "x_ratelimit_remaining": last_rate_limit_remaining,
                    "x_ratelimit_reset": last_rate_limit_reset
                })

                for ev in all_events:
                    event_rows.append(flatten_event(ev, artist))

                print(
                    f"  OK FINAL | n_shows_2024={n_shows_2024} | "
                    f"n_shows_2025={n_shows_2025} | "
                    f"requests={n_requests_for_artist}"
                )
                finished_ok = True
                break

            offset += LIMIT

        # si no terminó ok y tampoco guardó error explícito, no hacemos nada extra
        # porque ya quedó registrado dentro del loop

    except Exception as e:
        summary_rows.append({
            "downloaded_at_utc": now_utc_iso(),
            "chartmetric_id": artist_id,
            "artist_name": artist_name,
            "ok": False,
            "status_code": "",
            "error_text": repr(e),
            "raw_json_path": "",
            "n_shows_2024": "",
            "n_shows_2025": "",
            "n_requests_live_2024_2025": n_requests_for_artist,
            "x_ratelimit_limit": last_rate_limit_limit,
            "x_ratelimit_remaining": last_rate_limit_remaining,
            "x_ratelimit_reset": last_rate_limit_reset
        })
        print(f"  EXCEPTION: {e!r}")

# =========================
# SAVE OUTPUTS
# =========================
summary_fields = [
    "downloaded_at_utc",
    "chartmetric_id",
    "artist_name",
    "ok",
    "status_code",
    "error_text",
    "raw_json_path",
    "n_shows_2024",
    "n_shows_2025",
    "n_requests_live_2024_2025",
    "x_ratelimit_limit",
    "x_ratelimit_remaining",
    "x_ratelimit_reset"
]
write_csv_rows(SUMMARY_CSV, summary_rows, summary_fields)

if event_rows:
    event_fields = list(event_rows[0].keys())
    write_csv_rows(EVENTS_CSV, event_rows, event_fields)

print("\nListo.")
print(f"Resumen: {SUMMARY_CSV}")
print(f"Event-level: {EVENTS_CSV}")
print(f"Raw JSON dir: {RAW_DIR}")
print(f"Nuevos OK: {sum(1 for r in summary_rows if r['ok'])} / {len(summary_rows)}")

Total artistas en metadata: 5600
Ya descargados liveevents ok=True: 5532
Pendientes: 68
Se descargarán ahora: 1

Reusando access_token ya obtenido en esta sesión.

[1/1] Ragie Ban (4174234)
  status=402 error

Listo.
Resumen: chartmetric_liveevents_2024_2025_cohorte2\liveevents_2024_2025_summary_cohorte2.csv
Event-level: chartmetric_liveevents_2024_2025_cohorte2\liveevents_2024_2025_eventlevel_cohorte2.csv
Raw JSON dir: chartmetric_liveevents_2024_2025_cohorte2\raw_json_cohorte2
Nuevos OK: 0 / 1


## identifico filas duplicadas live

In [34]:
import csv
from pathlib import Path
from collections import defaultdict

INPUT_SUMMARY = Path("chartmetric_liveevents_2024_2025_cohorte2/liveevents_2024_2025_summary_cohorte2.csv")

def to_int(value, default=0):
    try:
        return int(float(str(value).strip()))
    except:
        return default

if not INPUT_SUMMARY.exists():
    raise FileNotFoundError(f"No existe el archivo: {INPUT_SUMMARY}")

with open(INPUT_SUMMARY, "r", encoding="utf-8-sig", newline="") as f:
    rows = list(csv.DictReader(f))

if not rows:
    raise ValueError("El summary está vacío.")

rows_by_id = defaultdict(list)

for row in rows:
    cmid = to_int(row.get("chartmetric_id", ""))
    if cmid:
        rows_by_id[cmid].append(row)

duplicates = {cmid: group for cmid, group in rows_by_id.items() if len(group) > 1}

total_ids = len(rows_by_id)
total_duplicates = sum(len(group) - 1 for group in duplicates.values())

print("=== DETECCIÓN DE DUPLICADOS ===")
print("Archivo:", INPUT_SUMMARY)
print("Filas totales:", len(rows))
print("IDs únicos:", total_ids)
print("Duplicados totales:", total_duplicates)
print("Artistas con duplicados:", len(duplicates))

if not duplicates:
    print("\nNo se detectaron duplicados.")
else:
    print("\nPrimeros 20 artistas con duplicados:")
    for cmid, group in list(duplicates.items())[:20]:
        print(f"{cmid} | repeticiones: {len(group)} | artist_name: {group[-1].get('artist_name', '')}")

=== DETECCIÓN DE DUPLICADOS ===
Archivo: chartmetric_liveevents_2024_2025_cohorte2\liveevents_2024_2025_summary_cohorte2.csv
Filas totales: 5550
IDs únicos: 5550
Duplicados totales: 0
Artistas con duplicados: 0

No se detectaron duplicados.


In [ ]:
import csv
from collections import defaultdict

SUMMARY_FILE = "chartmetric_liveevents_2024_2025_cohorte2/liveevents_2024_2025_summary_cohorte2.csv"

def to_int(value, default=0):
    try:
        return int(float(str(value).strip()))
    except:
        return default

# Leer filas
with open(SUMMARY_FILE, "r", encoding="utf-8-sig", newline="") as f:
    rows = list(csv.DictReader(f))

# Agrupar por chartmetric_id
rows_by_id = defaultdict(list)

for row in rows:
    cmid = to_int(row.get("chartmetric_id", ""))
    if cmid:
        rows_by_id[cmid].append(row)

# Detectar duplicados
duplicates = {k: v for k, v in rows_by_id.items() if len(v) > 1}

print("=== DUPLICADOS DETECTADOS COHORTE2 ===")
print("Cantidad de artistas duplicados:", len(duplicates))
print()

# Mostrar detalle
for cmid, group in duplicates.items():
    print("======================================")
    print(f"chartmetric_id: {cmid}")
    print(f"repeticiones: {len(group)}")
    
    for i, row in enumerate(group, start=1):
        print(f"\n--- fila {i} ---")
        print("artist_name:", row.get("artist_name"))
        print("ok:", row.get("ok"))
        print("status_code:", row.get("status_code"))
        print("n_shows_2024:", row.get("n_shows_2024"))
        print("n_shows_2025:", row.get("n_shows_2025"))
        print("n_requests:", row.get("n_requests_live_2024_2025"))
        print("downloaded_at:", row.get("downloaded_at_utc"))
    
    print()

=== DUPLICADOS DETECTADOS COHORTE2 ===
Cantidad de artistas duplicados: 0



## limpio filas duplicadas live

In [ ]:
import csv
from pathlib import Path
from datetime import datetime

INPUT_SUMMARY = Path("chartmetric_liveevents_2024_2025_cohorte2/liveevents_2024_2025_summary_cohorte2.csv")
OUTPUT_SUMMARY = Path("chartmetric_liveevents_2024_2025_cohorte2/liveevents_2024_2025_summary_dedup_cohorte2.csv")

def to_int(value, default=0):
    try:
        return int(float(str(value).strip()))
    except:
        return default

def count_unique_ids(rows, id_col="chartmetric_id"):
    ids = []
    for row in rows:
        cmid = to_int(row.get(id_col, ""))
        if cmid:
            ids.append(cmid)
    return len(set(ids)), len(ids) - len(set(ids))

if not INPUT_SUMMARY.exists():
    raise FileNotFoundError(f"No existe el archivo de entrada: {INPUT_SUMMARY}")

with open(INPUT_SUMMARY, "r", encoding="utf-8-sig", newline="") as f:
    rows = list(csv.DictReader(f))

if not rows:
    raise ValueError("El summary está vacío.")

fieldnames = list(rows[0].keys())

latest_by_id = {}
for row in rows:
    cmid = to_int(row.get("chartmetric_id", ""))
    if cmid:
        latest_by_id[cmid] = row

dedup_rows = list(latest_by_id.values())

if not dedup_rows:
    raise ValueError("La deduplicación produjo 0 filas. Proceso cancelado.")

with open(OUTPUT_SUMMARY, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(dedup_rows)

with open(OUTPUT_SUMMARY, "r", encoding="utf-8-sig", newline="") as f:
    check_rows = list(csv.DictReader(f))

if not check_rows:
    raise ValueError("El archivo deduplicado se escribió vacío. Proceso cancelado.")

unique_ids_before, dup_before = count_unique_ids(rows)
unique_ids_after, dup_after = count_unique_ids(check_rows)

print("=== LIMPIEZA SUMMARY ===")
print("Filas originales:", len(rows))
print("IDs únicos originales:", unique_ids_before)
print("Duplicados originales:", dup_before)
print("Filas deduplicadas:", len(check_rows))
print("IDs únicos deduplicados:", unique_ids_after)
print("Duplicados deduplicados:", dup_after)
print("Archivo deduplicado creado:", OUTPUT_SUMMARY)

if dup_after != 0:
    raise ValueError("El archivo deduplicado todavía tiene duplicados. Proceso cancelado.")

=== LIMPIEZA SUMMARY ===
Filas originales: 3497
IDs únicos originales: 3496
Duplicados originales: 1
Filas deduplicadas: 3496
IDs únicos deduplicados: 3496
Duplicados deduplicados: 0
Archivo deduplicado creado: chartmetric_liveevents_2024_2025_cohorte2\liveevents_2024_2025_summary_dedup_cohorte2.csv


## reasigno filenames

In [ ]:
import shutil
from pathlib import Path
from datetime import datetime
import csv

BASE_DIR = Path("chartmetric_liveevents_2024_2025_cohorte2")

original = BASE_DIR / "liveevents_2024_2025_summary_cohorte2.csv"
dedup = BASE_DIR / "liveevents_2024_2025_summary_dedup_cohorte2.csv"

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup = BASE_DIR / f"liveevents_2024_2025_summary_backup_{timestamp}_cohorte2.csv"

def count_rows(path):
    with open(path, "r", encoding="utf-8-sig", newline="") as f:
        rows = list(csv.DictReader(f))
    return len(rows)

if not dedup.exists():
    raise FileNotFoundError(f"No existe el archivo deduplicado: {dedup}")

dedup_rows = count_rows(dedup)
if dedup_rows == 0:
    raise ValueError("El archivo deduplicado está vacío. Reemplazo cancelado.")

if original.exists():
    shutil.copy2(original, backup)
    print("Backup creado:", backup.name)
else:
    print("No existe archivo original. El deduplicado pasará a ser el oficial sin backup previo.")

shutil.copy2(dedup, original)
print("Archivo deduplicado copiado como oficial:", original.name)

final_rows = count_rows(original)
if final_rows != dedup_rows:
    raise ValueError(
        f"Verificación fallida: original final={final_rows}, dedup={dedup_rows}. "
        "El backup quedó disponible para recuperación."
    )

print("=== REEMPLAZO COMPLETADO ===")
print("Filas en dedup:", dedup_rows)
print("Filas en original final:", final_rows)
print("Proceso terminado.")

Backup creado: liveevents_2024_2025_summary_backup_20260401_105000_cohorte2.csv
Archivo deduplicado copiado como oficial: liveevents_2024_2025_summary_cohorte2.csv
=== REEMPLAZO COMPLETADO ===
Filas en dedup: 3496
Filas en original final: 3496
Proceso terminado.


# control cohorte 2

In [35]:
import csv
from pathlib import Path

# =========================
# CONFIG
# =========================
METADATA_FILE = "chartmetric_metadata_cohorte2/metadata_flat_cohorte2.csv"
LIVE_SUMMARY_FILE = "chartmetric_liveevents_2024_2025_cohorte2/liveevents_2024_2025_summary_cohorte2.csv"


# =========================
# HELPERS
# =========================
def read_csv_rows(path):
    with open(path, "r", encoding="utf-8-sig", newline="") as f:
        return list(csv.DictReader(f))

def to_int(value, default=0):
    try:
        return int(float(str(value).strip()))
    except:
        return default

def to_bool_str(value):
    return str(value).strip().lower() == "true"

# =========================
# READ FILES
# =========================
metadata_rows = read_csv_rows(METADATA_FILE)
live_rows = read_csv_rows(LIVE_SUMMARY_FILE)

# =========================
# TOTAL COHORT FROM METADATA
# =========================
metadata_ids = []
seen_meta = set()

for row in metadata_rows:
    cmid = to_int(row.get("requested_chartmetric_id", ""))
    if cmid and cmid not in seen_meta:
        seen_meta.add(cmid)
        metadata_ids.append(cmid)

metadata_ids_set = set(metadata_ids)

# =========================
# LIVE EVENTS STATUS
# Si un artista aparece varias veces en el summary,
# nos quedamos con la última fila.
# =========================
live_by_id = {}

for row in live_rows:
    cmid = to_int(row.get("chartmetric_id", ""))
    if cmid:
        live_by_id[cmid] = row

live_ids_set = set(live_by_id.keys())

ok_ids = set()
error_ids = set()
pagination_ids = set()
pagination_ok_ids = set()
zero_show_ok_ids = set()
positive_show_ok_ids = set()

for cmid, row in live_by_id.items():
    ok = to_bool_str(row.get("ok", ""))
    n_requests = to_int(row.get("n_requests_live_2024_2025", "0"))
    n_2024 = to_int(row.get("n_shows_2024", "0"))
    n_2025 = to_int(row.get("n_shows_2025", "0"))
    total_shows = n_2024 + n_2025

    if ok:
        ok_ids.add(cmid)
        if total_shows == 0:
            zero_show_ok_ids.add(cmid)
        else:
            positive_show_ok_ids.add(cmid)
    else:
        error_ids.add(cmid)

    if n_requests > 1:
        pagination_ids.add(cmid)
        if ok:
            pagination_ok_ids.add(cmid)

pending_ids = metadata_ids_set - ok_ids
pending_because_error_ids = error_ids & pending_ids

# =========================
# PRINT MAIN STATUS
# =========================
print("=== STATUS GENERAL ===")
print("Total cohorte metadata:", len(metadata_ids_set))
print("Total filas live summary:", len(live_rows))
print("Artistas únicos vistos en live summary:", len(live_ids_set))
print("OK live events:", len(ok_ids))
print("Errores live events:", len(error_ids))
print("Pendientes live events:", len(pending_ids))

print("\n=== COBERTURA DE SHOWS ===")
print("OK con 0 shows (2024+2025):", len(zero_show_ok_ids))
print("OK con >0 shows (2024+2025):", len(positive_show_ok_ids))

print("\n=== PAGINACIÓN ===")
print("Artistas que requirieron más de 1 request:", len(pagination_ids))
print("De esos, resueltos OK:", len(pagination_ok_ids))
print("De esos, NO resueltos OK:", len(pagination_ids - pagination_ok_ids))

# =========================
# LISTAS ÚTILES PARA MAÑANA
# =========================
print("\n=== PENDIENTES (primeros 20) ===")
pending_preview = sorted(list(pending_ids))[:20]
print(pending_preview)

print("\n=== ERRORES / REINTENTAR (primeros 20) ===")
retry_preview = sorted(list(pending_because_error_ids))[:20]
print(retry_preview)

# =========================
# TOP ARTISTAS CON MÁS REQUESTS
# =========================
print("\n=== TOP 20 artistas por cantidad de requests live ===")
top_requests = []

for cmid, row in live_by_id.items():
    top_requests.append({
        "chartmetric_id": cmid,
        "artist_name": row.get("artist_name", ""),
        "ok": row.get("ok", ""),
        "n_requests_live_2024_2025": to_int(row.get("n_requests_live_2024_2025", "0")),
        "n_shows_2024": to_int(row.get("n_shows_2024", "0")),
        "n_shows_2025": to_int(row.get("n_shows_2025", "0")),
    })

top_requests = sorted(
    top_requests,
    key=lambda x: (
        x["n_requests_live_2024_2025"],
        x["n_shows_2024"] + x["n_shows_2025"]
    ),
    reverse=True
)

for row in top_requests[:20]:
    print(row)



=== STATUS GENERAL ===
Total cohorte metadata: 5600
Total filas live summary: 5550
Artistas únicos vistos en live summary: 5550
OK live events: 5532
Errores live events: 18
Pendientes live events: 68

=== COBERTURA DE SHOWS ===
OK con 0 shows (2024+2025): 2513
OK con >0 shows (2024+2025): 3019

=== PAGINACIÓN ===
Artistas que requirieron más de 1 request: 103
De esos, resueltos OK: 103
De esos, NO resueltos OK: 0

=== PENDIENTES (primeros 20) ===
[61, 490, 817, 1261, 1584, 2609, 2756, 2845, 3302, 4178, 4279, 4459, 10189, 17896, 35116, 65156, 69569, 81146, 89084, 101283]

=== ERRORES / REINTENTAR (primeros 20) ===
[490, 1261, 2756, 10189, 35116, 81146, 129808, 195130, 206575, 207536, 209828, 249933, 465564, 475535, 808651, 3567628, 3676427, 4174234]

=== TOP 20 artistas por cantidad de requests live ===
{'chartmetric_id': 141706, 'artist_name': 'Cem Adrian', 'ok': 'True', 'n_requests_live_2024_2025': 2, 'n_shows_2024': 57, 'n_shows_2025': 121}
{'chartmetric_id': 2207, 'artist_name': 'Ro